# EEG sleep foundation — train on a cloud GPU (Colab or Kaggle)

Track 03 (Sleep Onset). This notebook rebuilds the project and trains the
submissions. It detects whether it runs on Colab or Kaggle.

**Colab**
1. **Runtime → Change runtime type → GPU** (T4 is enough), then **Runtime → Run all**.
2. The Muse data (~1.1 GB) is downloaded from NEMAR on the first run and cached
   in Google Drive, so later sessions only extract it.
3. Each finished build (`build/<label>/`, `dist/<label>.zip`, the log) is copied
   to Drive right away, so a disconnect does not lose finished runs.

**Kaggle**
1. In the notebook's **Settings** panel: **Accelerator → GPU T4 x2** (not P100:
   recent PyTorch builds no longer support it), and **Internet → on** (needs a
   phone-verified Kaggle account). Only one of the two GPUs is used.
2. **Run all** interactively, or **Save Version → Save & Run All** to run it
   in the background without keeping the browser open (up to 12 h).
3. Each finished build is copied to `/kaggle/working/results/`, which Kaggle
   keeps as the notebook's **Output**: download the ZIPs from there. The Muse
   data is downloaded again in every session (a few minutes).

Generated by `scripts/make_colab_notebook.py`; regenerate it after editing the
solver locally.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    # Fine for the external-data prep (RUN_EXTERNAL_PREP with
    # RUN_EXPERIMENTS = False); training runs stop later without a GPU.
    print("WARNING: no GPU. Only the external-data prep can run in this session.")

## Settings

In [ ]:
from pathlib import Path

import os
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = not ON_KAGGLE and os.path.exists("/content")
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other")

# The working copy (code + 2 GB of data) lives outside /kaggle/working, so it
# is not saved as Kaggle output; only results/ is.
if ON_KAGGLE:
    BASE = Path("/kaggle/temp") if os.path.isdir("/kaggle/temp") else Path("/tmp")
else:
    BASE = Path("/content") if ON_COLAB else Path.home()
PROJECT = BASE / "EEG_sleep_foundation"
COMPETITION_REPO = "https://github.com/neural-interfaces26/2026-competition.git"
COMPETITION_COMMIT = "a66bc23d1a0e686b895c093a642b45c74970758b"   # the harness version used locally

USE_DRIVE = ON_COLAB    # Colab: cache the data and save the results in Google Drive
DRIVE_DIR = Path("/content/drive/MyDrive/EEG_sleep_foundation")
# Where finished runs are saved (and restored from on a re-run).
KAGGLE_RESULTS = Path("/kaggle/working/results")

# External pseudo-Muse PSG windows (Sleep-EDF + HMC, scripts/prep_external.py).
# RUN_EXTERNAL_PREP = True downloads and cuts them in this session (~1 h,
# ~25 GB streamed, ~250 MB result in /kaggle/working); afterwards save that
# file as a private Kaggle Dataset and attach it ("Add Input") so later
# sessions find it under /kaggle/input and skip the download.
RUN_EXTERNAL_PREP = False
EXT_OUT = (Path("/kaggle/working") if ON_KAGGLE else PROJECT / "data") / "external_windows.npz"
_found = sorted(Path("/kaggle/input").glob("**/external_windows.npz")) if ON_KAGGLE else []
EXT_FILE = (_found[0] if _found else EXT_OUT if (RUN_EXTERNAL_PREP or EXT_OUT.exists())
            else None)
print("external windows:", EXT_FILE or "none (round 8 skips its external runs)")

In [ ]:
# Drive can only be mounted by the Google account running this Colab session.
# Picking a different account in the popup fails with "credential
# propagation was unsuccessful": open this notebook while signed in to Colab
# as the account whose Drive you want to use.
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as err:
        USE_DRIVE = False
        print(f"Drive not mounted ({err}).\n"
              "Continuing WITHOUT Drive: the data is downloaded again and "
              "results are not saved if the runtime disconnects. Download "
              "them with the last cell.\n"
              "To use Drive, run this notebook in Colab signed in as the "
              "account that owns that Drive.")
elif ON_KAGGLE:
    print("Kaggle: results are saved to", KAGGLE_RESULTS)

## Code: official harness + this project's files

In [ ]:
import subprocess
if not (PROJECT / "competition").exists():
    subprocess.run(["git", "clone", "-q", COMPETITION_REPO,
                    str(PROJECT / "competition")], check=True)
subprocess.run(["git", "-C", str(PROJECT / "competition"), "checkout", "-q",
                COMPETITION_COMMIT], check=True)
for sub in ("submission", "scripts", "build", "dist"):
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)
%cd {PROJECT}

In [ ]:
%%writefile submission/cnn_gru.py
"""Track 03 (Sleep Onset) submission: an ensemble of spectral-feature -> GRU
models that each output a time-to-N2 distribution.

Each 5 s window is summarised by hand-crafted spectral features:
- log band powers (delta, theta, alpha, sigma, beta, high/EMG), absolute and
  relative, and the theta/alpha ratio: the classic sleep-scoring markers;
- with ``feats=2`` (default) also drowsiness markers: slow (< 0.6 Hz) power
  per channel and on the AF7-AF8 derivation (slow eye movements), 0.6-4 Hz
  power on AF7-AF8, spectral entropy and 95 % spectral edge per channel;
- with ``feats=3`` also movement / arousal markers: fraction of clipped
  samples, log peak amplitude and log line length (movements and arousals
  pull a drowsy sleeper back to wake).
Optionally (``use_cnn``, off by default since the bottleneck experiments
showed it does not help with this much data) a 1-D CNN on the raw signal
adds learned waveform features.
These, plus the time elapsed since the recording started, feed a GRU that
keeps its state across the windows of a recording (streamed evaluation: one
window per ``predict`` call, ``reset_state`` per recording).

Per-sleeper adaptation (``adapt``): the features are also given relative to
the recording's running mean, so each window is judged against the same
sleeper's history; with ``feats=2`` also relative to a ~1 min EMA (the
short-term change) and the EMA relative to the running mean (the trend);
with ``feats=3`` also a decaying peak of each movement marker's deviation
from the running mean over ~1 and ~5 min, so a recent movement stays
visible for a few minutes.
All statistics are causal, over the windows seen so far.

Preprocessing v2 (``SLEEP_PREP=2``): the band powers skip 48-52 Hz (the
recordings carry 50 Hz mains noise, which otherwise leaks into the 30-60 Hz
band and every relative power) and use a Welch estimate (2 s Hann segments,
1 s hop) instead of one 5 s periodogram, for less variance.

Autoencoder features (``SLEEP_AE``): a denoising MLP autoencoder is first
trained, without labels, on the training windows' log spectra (32
log-spaced bins, 0.4-47 Hz, on the 4 channels and AF7-AF8); its frozen
encoder's 32-dim code replaces the hand-crafted features (``ae=1``) or is
added to them (``ae=2``). The per-recording adaptation applies to it as to
the hand-crafted features. Only the encoder ships with the weights.

Staging output (``SLEEP_STAGE_WEIGHT`` > 0, training only): a second head on
the GRU predicts YASA's sleep stage of each window's 30 s epoch (wake /
drowsy = N1 + REM / N2+), soft labels distilled from YASA run offline on the
full recordings (scripts/yasa_check.py -> scripts/make_yasa_teacher.py ->
data/yasa_teacher.npz). It only shapes the GRU's state during training;
predictions come from the time-to-N2 head alone.

The head outputs a distribution over the time to the first N2 epoch, in
5 s bins (0, 5, ..., 595, 600 = capped). The ensemble averages the members'
distributions; ``p[0]`` is the hazard (probability that N2 starts within the
next window) and the prediction is the ``q``-quantile of the distribution,
with ``q`` tuned on held-out subjects.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import os
import time

import numpy as np
import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

BIN_S = 5.0
N_BINS = int(CAP_S / BIN_S) + 1          # 0, 5, ..., 600 s
CLIP_UV = 500.0
BANDS = ((0.5, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 15.0), (15.0, 30.0),
         (30.0, 60.0))
# Metric ranges and severity weights (objective.py).
RANGE_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
RANGE_WEIGHTS = (10.0, 5.0, 3.0, 1.0)

CNN_CHANNELS = (32, 64, 96, 128)
BP_DIM = 64
GRU_HIDDEN = int(os.environ.get("SLEEP_GRU_HIDDEN", 128))
GRU_LAYERS = 2
DROPOUT = 0.2

# Training-time choices; inference reads them back from weights.pt.
N_SEEDS = int(os.environ.get("SLEEP_N_SEEDS", 5))
ADAPT = os.environ.get("SLEEP_ADAPT", "1") == "1"
# Raw-signal CNN branch (off: it did not help in the bottleneck experiments)
# and the feature set (1 = band powers only, 2 = + drowsiness and trend).
USE_CNN = os.environ.get("SLEEP_USE_CNN", "0") == "1"
FEATS = int(os.environ.get("SLEEP_FEATS", 2))
# Also train on NeuralBench's own validation sleepers of the public Muse
# training release, which the harness's train loader leaves out.
EXTRA_DATA = os.environ.get("SLEEP_EXTRA_DATA", "1") == "1"
# Diagnostics (see colab notebook): train on a fraction of the training
# sleepers, change the members' seed. The validation sleepers do not depend
# on any of these settings, so runs stay comparable.
TRAIN_FRAC = float(os.environ.get("SLEEP_TRAIN_FRAC", 1.0))
MEMBER_SEED = int(os.environ.get("SLEEP_SEED", 0))
EMA_S = float(os.environ.get("SLEEP_EMA_S", 60.0))  # short-term EMA, s
N_MOVE = 3                               # movement markers (feats >= 3)
PEAK_S = (60.0, 300.0)                   # decaying-peak horizons (feats >= 3)
# Staging output: loss weight (0 = off) and the YASA teacher labels.
STAGE_WEIGHT = float(os.environ.get("SLEEP_STAGE_WEIGHT", 0.0))
PREP = int(os.environ.get("SLEEP_PREP", 1))     # 2: notch + Welch
AE_MODE = int(os.environ.get("SLEEP_AE", 0))     # 1 replace, 2 add
AE_DIM = int(os.environ.get("SLEEP_AE_DIM", 32))  # autoencoder code size
AE_HIDDEN = 128
AE_EPOCHS = int(os.environ.get("SLEEP_AE_EPOCHS", 20))
LINE_HZ = (48.0, 52.0)                   # mains band skipped with PREP >= 2
TEACHER_FILE = os.environ.get("SLEEP_TEACHER", os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data",
    "yasa_teacher.npz"))
N_STAGES = 3                             # wake, drowsy (N1 + REM), N2+
# Cross-validated all-data training (final builds): with SLEEP_CV_FOLDS = K
# > 0 the ensemble has K members; member k trains on every recording except
# the sleepers of fold k, early-stops on fold k, and the quantile is chosen
# on the out-of-fold predictions of all recordings. SLEEP_USE_TEST=1 also
# trains on the 40 public warm-up test recordings (the warm-up is over; the
# local test score is then meaningless).
CV_FOLDS = int(os.environ.get("SLEEP_CV_FOLDS", 0))
# Members per fold (repeats with seeds SLEEP_SEED, SLEEP_SEED + 1, ...): the
# ensemble then has CV_FOLDS * CV_SEEDS members, and each recording's
# out-of-fold distribution is the mean over its fold's members.
CV_SEEDS = int(os.environ.get("SLEEP_CV_SEEDS", 1))
USE_TEST = os.environ.get("SLEEP_USE_TEST", "0") == "1"

EPOCHS = int(os.environ.get("SLEEP_EPOCHS", 30))
PATIENCE = 8
LR = float(os.environ.get("SLEEP_LR", 1e-3))
WEIGHT_DECAY = 1e-4
RECS_PER_BATCH = 16
CHUNK = int(os.environ.get("SLEEP_CHUNK", 120))  # TBPTT length, windows
VAL_FRACTION = 0.15
LABEL_SIGMA = float(os.environ.get("SLEEP_LABEL_SIGMA", 1.5))  # in bins
L1_WEIGHT = float(os.environ.get("SLEEP_L1", 0.1))
# External pseudo-Muse PSG windows (scripts/prep_external.py), CV mode only:
# added to every fold's training set with their loss weight scaled by
# EXT_WEIGHT; a domain flag input tells them apart (always 0 at inference).
EXTERNAL = os.environ.get("SLEEP_EXTERNAL", "")
EXT_WEIGHT = float(os.environ.get("SLEEP_EXT_WEIGHT", 0.5))
# Elapsed-time buckets (minutes) of the per-bucket quantile ``q_time``.
TIME_EDGES_MIN = (2.0, 5.0, 10.0, 20.0)
QUANTILES = (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6)
SEED = 0                                 # validation split seed (fixed)


def preprocess(X):
    """Raw microvolts (..., C, T) -> per-window DC removed, artifacts clipped."""
    X = X - X.mean(dim=-1, keepdim=True)
    return X.clamp(-CLIP_UV, CLIP_UV)


def quantile_seconds(p, q):
    """Distributions (..., K) over the 5 s bins -> their q-quantile, seconds."""
    cdf = p.cumsum(-1).contiguous()
    k = torch.searchsorted(cdf, torch.full(cdf.shape[:-1] + (1,), float(q),
                                           device=cdf.device))
    return (k.clamp_max(N_BINS - 1).squeeze(-1) * BIN_S).float()


class SleepOnsetCNNGRU(nn.Module):
    """One member: (B, L, C, T) sequences -> logits (B, L, K) over bins."""

    def __init__(self, n_chans, n_times, sfreq, adapt=ADAPT, use_cnn=USE_CNN,
                 feats=FEATS, stage=STAGE_WEIGHT > 0, prep=PREP, ae=AE_MODE,
                 hidden=GRU_HIDDEN, ae_dim=AE_DIM, ema_s=EMA_S, dom=False):
        super().__init__()
        self.prep, self.ae = int(prep), int(ae)
        self.hidden, self.dom = int(hidden), bool(dom)
        self.register_buffer("ema_s", torch.tensor(float(ema_s)))
        self.register_buffer("dom_on", torch.tensor(self.dom))
        self.register_buffer("prep_v", torch.tensor(self.prep))
        self.register_buffer("ae_mode", torch.tensor(self.ae))
        self.n_chans, self.n_times, self.sfreq = n_chans, n_times, sfreq
        self.window_s = n_times / sfreq
        self.use_adapt = bool(adapt)
        self.use_cnn = bool(use_cnn)
        self.n_feats = int(feats)
        self.ema_alpha = min(1.0, self.window_s / float(ema_s))
        self.register_buffer("adapt", torch.tensor(self.use_adapt))
        self.register_buffer("cnn_on", torch.tensor(self.use_cnn))
        self.register_buffer("feats", torch.tensor(self.n_feats))
        self.register_buffer("chan_std", torch.ones(n_chans))
        self.register_buffer(
            "bin_values", torch.arange(N_BINS, dtype=torch.float32) * BIN_S)

        # Band masks over the rFFT bins, from the actual sampling rate.
        freqs = torch.fft.rfftfreq(n_times, d=1.0 / sfreq)
        nyq = sfreq / 2
        # Band powers: one 5 s periodogram (prep 1) or Welch over 2 s Hann
        # segments with a 1 s hop, skipping the mains band (prep 2).
        self.seg = int(round(2 * sfreq)) if self.prep >= 2 else n_times
        self.hop = int(round(sfreq))
        if self.seg > n_times:
            self.seg = n_times
        bfreqs = torch.fft.rfftfreq(self.seg, d=1.0 / sfreq)
        keep = torch.ones_like(bfreqs, dtype=torch.bool)
        if self.prep >= 2:
            keep = (bfreqs < LINE_HZ[0]) | (bfreqs > LINE_HZ[1])
        masks = torch.stack([((bfreqs >= lo) & (bfreqs < min(hi, nyq))
                              & keep).float() for lo, hi in BANDS])
        self.register_buffer("band_masks", masks, persistent=False)
        self.register_buffer("seg_hann", torch.hann_window(self.seg),
                             persistent=False)
        self.register_buffer("hann", torch.hann_window(n_times),
                             persistent=False)
        self.register_buffer("freqs", freqs, persistent=False)
        self.register_buffer("slow_mask", ((freqs > 0) & (freqs < 0.6)).float(),
                             persistent=False)
        self.register_buffer("eog_mask", ((freqs >= 0.6) & (freqs < 4.0))
                             .float(), persistent=False)
        self.register_buffer("broad_mask", ((freqs >= 0.5)
                                            & (freqs < min(30.0, nyq))),
                             persistent=False)
        if self.ae:
            # Log-spaced spectrum bins, at least 0.4 Hz wide, up to 47 Hz.
            edges, e = [0.4], 0.4
            while e < 47.0:
                e = min(47.0, max(e * 1.15, e + 0.4))
                edges.append(e)
            bins = torch.stack([((freqs >= lo) & (freqs < hi)).float()
                                for lo, hi in zip(edges[:-1], edges[1:])], 1)
            self.register_buffer("ae_bins", bins / bins.sum(0).clamp_min(1),
                                 persistent=False)
            ae_in = (n_chans + 1) * bins.shape[1]
            self.register_buffer("spec_mean", torch.zeros(ae_in))
            self.register_buffer("spec_std", torch.ones(ae_in))
            self.ae_enc = nn.Sequential(nn.Linear(ae_in, AE_HIDDEN), nn.GELU(),
                                        nn.Linear(AE_HIDDEN, int(ae_dim)))
            self.ae_enc.requires_grad_(False)   # trained beforehand, frozen
        else:
            self.ae_enc = None
        n_bp = self.band_powers(torch.zeros(1, n_chans, n_times)).shape[1]
        self.register_buffer("bp_mean", torch.zeros(n_bp))
        self.register_buffer("bp_std", torch.ones(n_bp))

        layers, in_ch = [], n_chans
        kernels = (max(3, int(sfreq // 2)) | 1, 9, 7, 5)
        for out_ch, k in zip(CNN_CHANNELS, kernels):
            layers += [nn.Conv1d(in_ch, out_ch, k, padding=k // 2, bias=False),
                       nn.BatchNorm1d(out_ch), nn.GELU(), nn.MaxPool1d(2)]
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers) if self.use_cnn else None
        self.use_ema = self.n_feats >= 2 or self.ae > 0
        self.use_peaks = self.n_feats >= 3 and self.ae != 1
        bp_in = n_bp * ((4 if self.use_ema else 2) if self.use_adapt
                        else 1)
        if self.use_adapt and self.use_peaks:
            bp_in += N_MOVE * len(PEAK_S)
        # Decaying peaks lose 1 normalised unit per PEAK_S seconds.
        self.peak_decay = tuple(self.window_s / h for h in PEAK_S)
        self.bp_proj = nn.Sequential(nn.Linear(bp_in, BP_DIM), nn.GELU())
        cnn_dim = 2 * CNN_CHANNELS[-1] if self.use_cnn else 0
        self.in_proj = nn.Sequential(
            nn.Linear(cnn_dim + BP_DIM + 1 + int(self.dom), self.hidden),
            nn.GELU(), nn.Dropout(DROPOUT))
        self.gru = nn.GRU(self.hidden, self.hidden, GRU_LAYERS, batch_first=True,
                          dropout=DROPOUT)
        self.head = nn.Sequential(nn.Dropout(DROPOUT),
                                  nn.Linear(self.hidden, N_BINS))
        # Training-only auxiliary head (see module docstring).
        self.stage_head = (nn.Sequential(nn.Dropout(DROPOUT),
                                         nn.Linear(self.hidden, N_STAGES))
                           if stage else None)
        self.reset_state()

    def ae_input(self, X):
        """Preprocessed (N, C, T) -> log binned spectra (N, (C + 1) * nb)."""
        a, b = (1, 2) if X.shape[1] >= 3 else (0, X.shape[1] - 1)
        X = torch.cat([X, (X[:, a] - X[:, b]).unsqueeze(1)], dim=1)
        spec = torch.fft.rfft(X * self.hann, dim=-1).abs() ** 2
        return torch.log(spec @ self.ae_bins + 1e-6).flatten(1)

    def band_powers(self, X):
        """Preprocessed (N, C, T) -> raw features (N, F): the autoencoder
        code and/or the hand-crafted features."""
        if not self.ae:
            return self.hand_features(X)
        code = self.ae_enc((self.ae_input(X) - self.spec_mean)
                           / self.spec_std)
        if self.ae == 1:
            return code
        return torch.cat([code, self.hand_features(X)], dim=1)

    def hand_features(self, X):
        """Preprocessed (N, C, T) -> hand-crafted log band-power features."""
        spec = torch.fft.rfft(X * self.hann, dim=-1).abs() ** 2
        if self.seg < X.shape[-1]:
            segs = X.unfold(-1, self.seg, self.hop)
            bspec = (torch.fft.rfft(segs * self.seg_hann, dim=-1).abs() ** 2
                     ).mean(-2)
        else:
            bspec = spec
        power = torch.einsum("nct,bt->ncb", bspec, self.band_masks)
        log_abs = torch.log(power + 1e-6)
        log_rel = torch.log(power / (power.sum(-1, keepdim=True) + 1e-6)
                            + 1e-6)
        theta_alpha = log_abs[..., 1] - log_abs[..., 2]
        out = [log_abs.flatten(1), log_rel.flatten(1), theta_alpha]
        if self.n_feats >= 2:
            # Drowsiness: slow eye movements (< 0.6 Hz, strongest on the
            # AF7-AF8 derivation, where horizontal eye movements are of
            # opposite polarity), spectral flattening (entropy, edge).
            out.append(torch.log(spec @ self.slow_mask + 1e-6))
            a, b = (1, 2) if X.shape[1] >= 3 else (0, X.shape[1] - 1)
            eog = torch.fft.rfft((X[:, a] - X[:, b]) * self.hann,
                                 dim=-1).abs() ** 2
            out.append(torch.stack([torch.log(eog @ self.slow_mask + 1e-6),
                                    torch.log(eog @ self.eog_mask + 1e-6)],
                                   dim=1))
            p = spec[..., self.broad_mask]
            p = p / (p.sum(-1, keepdim=True) + 1e-12)
            out.append(-(p * torch.log(p + 1e-12)).sum(-1)
                       / np.log(p.shape[-1]))
            edge = (p.cumsum(-1) >= 0.95).float().argmax(-1)
            out.append(self.freqs[self.broad_mask][edge] / 30.0)
        if self.n_feats >= 3:
            # Movement / arousal markers; the last N_MOVE features. X is
            # already clipped at CLIP_UV, so "clipped" = at that limit.
            a = X.abs()
            out.append(torch.stack([
                (a >= 0.99 * CLIP_UV).float().mean((1, 2)),
                torch.log(a.amax((1, 2)) + 1e-3),
                torch.log((X[..., 1:] - X[..., :-1]).abs().sum(-1).mean(-1)
                          + 1e-3)], dim=1))
        return torch.cat(out, dim=1)

    def encode(self, X, start_index, sums, dom=None):
        """Sequences (B, L, C, T) -> GRU inputs (B, L, H), new running sums.

        ``sums`` = (sum of band powers (B, F), sum of channel variances
        (B, C), window count (B, 1), short-term EMA of the band powers
        (B, F), decaying movement peaks (list of (B, N_MOVE) or None)) over
        the earlier windows of each recording, or None at its start.
        """
        B, L, C, T = X.shape
        Xf = preprocess(X.reshape(B * L, C, T))
        bp = ((self.band_powers(Xf) - self.bp_mean) / self.bp_std
              ).reshape(B, L, -1)
        if self.use_adapt:
            var = Xf.var(-1).reshape(B, L, C)
            if sums is None:
                sums = (bp.new_zeros(B, bp.shape[-1]), var.new_zeros(B, C),
                        var.new_zeros(B, 1), None, None)
            count = sums[2].unsqueeze(1) + torch.arange(
                1, L + 1, device=X.device).float().view(1, L, 1)
            cum_bp = sums[0].unsqueeze(1) + bp.cumsum(1)
            cum_var = sums[1].unsqueeze(1) + var.cumsum(1)
            mean = cum_bp / count
            rel = bp - mean
            scale = (cum_var / count).sqrt().clamp_min(
                0.25 * self.chan_std.view(1, 1, C))
            parts_bp = [bp, rel]
            ema = sums[3]
            if self.use_ema:
                emas = []
                for i in range(L):
                    ema = (bp[:, i] if ema is None
                           else ema + self.ema_alpha * (bp[:, i] - ema))
                    emas.append(ema)
                emas = torch.stack(emas, dim=1)
                parts_bp += [bp - emas, emas - mean]
            peaks = sums[4] if len(sums) > 4 else None
            if self.use_peaks:
                # Decaying peak of each movement marker's deviation from the
                # recording's running mean: jumps to a new movement, then
                # fades by 1 normalised unit per horizon.
                move = rel[..., -N_MOVE:]
                out_peaks = [[] for _ in self.peak_decay]
                for i in range(L):
                    x = move[:, i]
                    peaks = [x if peaks is None else
                             torch.maximum(x, pk - d)
                             for pk, d in zip(peaks or [None] * len(
                                 self.peak_decay), self.peak_decay)]
                    for j, pk in enumerate(peaks):
                        out_peaks[j].append(pk)
                parts_bp += [torch.stack(o, dim=1) for o in out_peaks]
            sums = (cum_bp[:, -1], cum_var[:, -1], count[:, -1], ema, peaks)
            bp_in = torch.cat(parts_bp, dim=-1)
        else:
            scale = self.chan_std.view(1, 1, C).expand(B, L, C)
            bp_in = bp
        parts = []
        if self.use_cnn:
            h = self.cnn(Xf / scale.reshape(B * L, C, 1))
            parts.append(torch.cat([h.mean(-1), h.std(-1)], dim=1
                                   ).reshape(B, L, -1))

        idx = torch.arange(L, device=X.device).float().unsqueeze(0)
        idx = idx + torch.as_tensor(start_index, device=X.device,
                                    dtype=torch.float32).reshape(-1, 1)
        t = torch.log1p(idx.expand(B, L) * self.window_s / 60.0).unsqueeze(-1)
        extra = [t]
        if self.dom:
            # Domain flag: 1 for external PSG recordings (training only).
            d = (torch.zeros(B, device=X.device) if dom is None
                 else dom.to(X.device).float())
            extra.append(d.view(B, 1, 1).expand(B, L, 1))
        z = self.in_proj(torch.cat(parts + [self.bp_proj(bp_in)] + extra,
                                   dim=-1))
        return z, sums

    def forward(self, X, start_index=0, state=None, return_stage=False,
                dom=None):
        """Sequences (B, L, C, T) of consecutive windows -> logits (B, L, K).

        ``start_index`` is the index of the first window in its recording
        (int or (B,) tensor) and ``state`` = (GRU state, running sums) is
        carried from the previous chunk of the same recordings. With
        ``return_stage`` the staging logits (B, L, 3) come second.
        """
        h, sums = state if state is not None else (None, None)
        z, sums = self.encode(X, start_index, sums, dom)
        out, h = self.gru(z, h)
        if return_stage:
            return self.head(out), self.stage_head(out), (h, sums)
        return self.head(out), (h, sums)

    # --- streamed inference ---------------------------------------------

    def step_proba(self, X):
        """Next windows (B, C, T) of the current recording -> p (B, K)."""
        # The B windows of a call are consecutive windows of one recording
        # (B = 1 in streamed evaluation).
        logits, self._state = self(X.float().unsqueeze(0), self._n,
                                   self._state)
        self._n += X.shape[0]
        return logits[0].softmax(-1)

    def reset_state(self):
        self._state = None
        self._n = 0


class SleepOnsetEnsemble(nn.Module):
    """Averages the members' time-to-N2 distributions; predicts a quantile."""

    def __init__(self, n_chans, n_times, sfreq, n_members=N_SEEDS,
                 adapt=ADAPT, use_cnn=USE_CNN, feats=FEATS,
                 stage=STAGE_WEIGHT > 0, prep=PREP, ae=AE_MODE,
                 configs=None, dom=False):
        super().__init__()
        # ``configs``: one dict of member settings per member (mixed
        # ensembles, see scripts/merge_builds.py); else n_members alike.
        if configs is None:
            configs = [dict(adapt=adapt, use_cnn=use_cnn, feats=feats,
                            stage=stage, prep=prep, ae=ae, dom=dom)
                       for _ in range(n_members)]
        self.members = nn.ModuleList(
            SleepOnsetCNNGRU(n_chans, n_times, sfreq, **c) for c in configs)
        self.register_buffer("quantile", torch.tensor(0.5))
        # Averaging weights of the members, and the quantile per elapsed-time
        # bucket (TIME_EDGES_MIN); both default to plain averaging / q.
        self.register_buffer("member_w", torch.ones(len(configs)))
        self.register_buffer("q_time", torch.full(
            (len(TIME_EDGES_MIN) + 1,), 0.5))

    def set_quantile(self, q):
        self.quantile.fill_(float(q))
        self.q_time.fill_(float(q))

    def q_for(self, start, n):
        """Quantiles (n,) of windows start .. start + n - 1 of a recording."""
        window_s = self.members[0].window_s
        minutes = (start + torch.arange(n, device=self.q_time.device)
                   ) * window_s / 60.0
        edges = torch.tensor(TIME_EDGES_MIN, device=self.q_time.device)
        return self.q_time[torch.bucketize(minutes, edges, right=True)]

    def _step_proba(self, X):
        w = self.member_w / self.member_w.sum()
        return (torch.stack([m.step_proba(X) for m in self.members])
                * w.view(-1, 1, 1)).sum(0)

    def _seconds(self, p, start):
        q = self.q_for(start, p.shape[0])
        cdf = p.cumsum(-1).contiguous()
        k = torch.searchsorted(cdf, q.view(-1, 1).to(cdf.dtype))
        return (k.clamp_max(N_BINS - 1).squeeze(-1) * BIN_S).float(
        ).clamp(0.0, CAP_S)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        start = self.members[0]._n
        p = self._step_proba(X)
        return self._seconds(p, start)

    @torch.inference_mode()
    def predict_dist(self, X):
        """(seconds, p over 5 s bins, hazard) for analysis; advances state."""
        self.eval()
        start = self.members[0]._n
        p = self._step_proba(X)
        return self._seconds(p, start), p, p[..., 0]

    def reset_state(self):
        for m in self.members:
            m.reset_state()


# --- training helpers (local only) ------------------------------------------


def _load_windows(dataset):
    """Read every window once, in dataset order, preprocessed, as float16."""
    from torch.utils.data import DataLoader
    Xs, ys, recs, onsets, subs = [], [], [], [], []
    t0 = time.time()
    for X, y, info in DataLoader(dataset, batch_size=512, shuffle=False):
        Xs.append(preprocess(X.float().cpu()).half())
        ys.append(y.float().cpu().reshape(-1))
        recs.append(np.asarray(info["record_id"]).reshape(-1))
        onsets.append(np.asarray(info["onset"], dtype=np.float64).reshape(-1))
        sid = info.get("subject_id", info["record_id"])
        subs.append(np.asarray(sid).reshape(-1))
    print(f"[CNN-GRU] loaded {sum(len(x) for x in Xs)} windows in "
          f"{time.time() - t0:.0f} s")
    return (torch.cat(Xs), torch.cat(ys), np.concatenate(recs),
            np.concatenate(onsets), np.concatenate(subs))


def _load_splits(train_dataset, splits):
    """Other splits of the public Muse release -> {split: (windows, dataset)}.

    The harness builds NeuralBench's train/val/test loaders but hands the
    solver only the train one. Its ``val`` split (20 % of the sleepers
    without a test recording, fixed by ``valid_random_state``) and ``test``
    split (the 40 public warm-up test recordings) are public training data
    too, so they are loaded here, for training only. Local training only:
    never runs on Codabench. Returns {} when unavailable (e.g. the Simulated
    dataset); a split that would overlap another one is skipped.
    """
    if not hasattr(train_dataset, "seg_ds") or not splits:
        return {}
    try:
        from benchopt.config import get_data_path
        from benchmark_utils.nb_task import load_task
        t0 = time.time()
        loaders, _ = load_task(
            "eeg", "_sleep_onset_stream",
            data_dir=get_data_path("neural_compet"), device="cpu",
            batch_size=512, target_transform=lambda y: y.reshape(()))
    except Exception as err:          # noqa: BLE001 - optional extra data
        print(f"[CNN-GRU] extra data unavailable ({err}); skipping")
        return {}
    seen = {seg.timeline for seg in train_dataset.seg_ds.segments}
    out = {}
    for split in splits:
        ds = loaders[split].dataset
        tl = {seg.timeline for seg in ds.seg_ds.segments}
        if seen & tl:
            print(f"[CNN-GRU] split {split} overlaps the data already "
                  f"loaded ({len(seen & tl)} recordings); skipping")
            continue
        seen |= tl
        print(f"[CNN-GRU] extra split {split}: {len(tl)} recordings "
              f"(set up in {time.time() - t0:.0f} s)")
        out[split] = (_load_windows(ds), ds)
    return out


def _extra_windows(train_dataset):
    """NeuralBench's validation split -> (windows, dataset), or None."""
    return _load_splits(train_dataset, ["val"]).get("val")


def _timeline_fields(dataset, n):
    """{subject, session, ...} of each of the first ``n`` windows, or None."""
    if not hasattr(dataset, "seg_ds"):
        return None
    segs = dataset.seg_ds.segments
    order = getattr(dataset, "index", np.arange(len(segs)))
    return [dict(kv.split("=") for kv in
                 segs[int(order[i])].timeline.split(":", 1)[1].split(","))
            for i in range(n)]


def _teacher_targets(dataset, onset):
    """YASA stage probabilities (N, 3) of each window + a found mask (N,).

    Window i of ``dataset`` starts ``onset[i]`` s into its recording (the
    harness windows tile each recording from 0 in 5 s steps), so it lies in
    the recording's 30 s YASA epoch ``floor(onset / 30)``. Recordings are
    matched by subject and session, parsed from the window's timeline.
    """
    n = len(onset)
    target = torch.zeros(n, N_STAGES)
    found = torch.zeros(n, dtype=torch.bool)
    all_fields = _timeline_fields(dataset, n)
    if all_fields is None:
        return target, found
    t = np.load(TEACHER_FILE)
    where = {k: (int(a), int(b)) for k, a, b in zip(t["keys"], t["start"],
                                                    t["n"])}
    proba = torch.as_tensor(t["proba"].astype(np.float32))
    for i, fields in enumerate(all_fields):
        key = f"{fields['subject']}/{fields['session']}"
        if key not in where:
            continue
        start, count = where[key]
        k = int(onset[i] // 30.0)
        if k < count:
            target[i] = proba[start + k]
            found[i] = True
    return target, found


def _recordings(y, rec, onset, sub, rng):
    """Window indices of each recording, in time order, plus its subject.

    Data without real recordings (the Simulated set: one record of random
    windows) is cut into pseudo-recordings sorted by decreasing latency.
    """
    out = []
    if len(np.unique(rec)) < 4:
        perm = rng.permutation(len(y))
        for i, part in enumerate(np.array_split(perm, 10)):
            part = part[np.argsort(-y[part].numpy(), kind="stable")]
            out.append((part, i))
        return out
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        out.append((idx[np.argsort(onset[idx], kind="stable")],
                    int(sub[idx[0]])))
    return out


def _window_weights(y):
    """Per-window loss weights of one recording, mimicking its W-bMAE.

    Each non-empty metric range gets its severity weight split over its
    windows; the recording's weights sum to 1.
    """
    edges = np.asarray(RANGE_EDGES)
    r = np.clip(np.digitize(y, edges[1:-1]), 0, len(RANGE_WEIGHTS) - 1)
    w = np.zeros(len(y))
    present = np.unique(r)
    total = sum(RANGE_WEIGHTS[b] for b in present)
    for b in present:
        mask = r == b
        w[mask] = RANGE_WEIGHTS[b] / total / mask.sum()
    return torch.as_tensor(w, dtype=torch.float32)


def _soft_labels(y, device):
    """Targets (N,) seconds -> Gaussian-smoothed bin distributions (N, K)."""
    centers = torch.arange(N_BINS, device=device).float()
    pos = (y.clamp(0, CAP_S) / BIN_S).unsqueeze(-1)
    logits = -0.5 * ((centers - pos) / LABEL_SIGMA) ** 2
    return logits.softmax(-1)


def _augment(X):
    """Per-recording amplitude scaling, noise and channel dropout.

    X: (B, L, C, T), preprocessed microvolts.
    """
    B, _, C, _ = X.shape
    scale = torch.empty(B, 1, 1, 1, device=X.device).uniform_(0.7, 1.3)
    X = X * scale + torch.randn_like(X) * 2.0
    drop = torch.rand(B, device=X.device) < 0.1
    if drop.any():
        ch = torch.randint(0, C, (B,), device=X.device)
        mask = torch.ones(B, 1, C, 1, device=X.device)
        mask[drop, 0, ch[drop], 0] = 0.0
        X = X * mask
    return X


def _batch_tensor(X_all, recs, start, stop, device):
    """Pad the windows [start, stop) of each recording into (B, L, C, T)."""
    L = min(stop, max(len(idx) for idx in recs)) - start
    X = torch.zeros(len(recs), L, *X_all.shape[1:], dtype=torch.float16)
    mask = torch.zeros(len(recs), L, dtype=torch.bool)
    for b, idx in enumerate(recs):
        part = idx[start:stop]
        X[b, :len(part)] = X_all[part]
        mask[b, :len(part)] = True
    return X.to(device).float(), mask.to(device)


def _detach(state):
    h, sums = state

    def det(s):
        if s is None:
            return None
        if isinstance(s, list):
            return [x.detach() for x in s]
        return s.detach()
    return h.detach(), None if sums is None else tuple(det(s) for s in sums)


@torch.no_grad()
def _sequence_proba(member, X_all, idx, device):
    """Distributions (L, K) of one whole recording, chunked, in eval mode."""
    member.eval()
    out, state = [], None
    for s in range(0, len(idx), 4 * CHUNK):
        X, _ = _batch_tensor(X_all, [idx], s, s + 4 * CHUNK, device)
        logits, state = member(X, s, state)
        out.append(logits[0].softmax(-1))
    return torch.cat(out)


def _scores(probas, y_all, recs):
    """W-bMAE averaged over recordings, for every candidate quantile.

    ``probas[i]`` is the (L, K) distribution of recording ``recs[i]``.
    """
    from benchmark_utils.metrics import binned_mae
    scores = {q: [] for q in QUANTILES}
    for p, (idx, _sub) in zip(probas, recs):
        y = y_all[idx].numpy()
        for q in QUANTILES:
            pred = quantile_seconds(p, q).cpu().numpy()
            scores[q].append(binned_mae(y, pred, RANGE_EDGES, RANGE_WEIGHTS))
    return {q: float(np.mean(s)) for q, s in scores.items()}


def _train_device(default):
    forced = os.environ.get("SLEEP_TRAIN_DEVICE")
    if forced:
        return forced
    if default == "cpu" and torch.backends.mps.is_available():
        return "mps"
    return default


def _train_member(model, seed, X_all, y_all, train_recs, val_recs, weights,
                  device, tag, stage_all=None, ext_mask=None):
    """Train one member in place; keep its best epoch on validation."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    opt = torch.optim.AdamW([p for p in model.parameters()
                             if p.requires_grad], lr=LR,
                            weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

    best, best_state, bad = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        order = rng.permutation(len(train_recs))
        # Batch recordings of similar length to limit padding.
        order = sorted(order, key=lambda i: len(train_recs[i][0])
                       + rng.uniform(0, 60))
        batches = [order[i:i + RECS_PER_BATCH]
                   for i in range(0, len(order), RECS_PER_BATCH)]
        rng.shuffle(batches)
        use_stage = stage_all is not None and model.stage_head is not None
        running, running_stage, n_rec = 0.0, 0.0, 0
        for batch in batches:
            idxs = [train_recs[i][0] for i in batch]
            max_len = max(len(idx) for idx in idxs)
            y = torch.zeros(len(idxs), max_len)
            w = torch.zeros(len(idxs), max_len)
            st = torch.zeros(len(idxs), max_len, N_STAGES)
            sm = torch.zeros(len(idxs), max_len)
            for b, idx in enumerate(idxs):
                y[b, :len(idx)] = y_all[idx]
                w[b, :len(idx)] = weights[id(idx)]
                if use_stage:
                    st[b, :len(idx)] = stage_all[0][idx]
                    sm[b, :len(idx)] = stage_all[1][idx].float()
            y, w = y.to(device), w.to(device)
            dom = (torch.tensor([bool(ext_mask[idx[0]]) for idx in idxs])
                   if ext_mask is not None and model.dom else None)
            st, sm = st.to(device), sm.to(device)
            state = None
            for s in range(0, max_len, CHUNK):
                X, mask = _batch_tensor(X_all, idxs, s, s + CHUNK, device)
                if use_stage:
                    logits, stage_logits, state = model(_augment(X), s, state,
                                                        return_stage=True,
                                                        dom=dom)
                else:
                    logits, state = model(_augment(X), s, state, dom=dom)
                state = _detach(state)
                yc, wc = y[:, s:s + CHUNK], w[:, s:s + CHUNK] * mask
                logp = logits.log_softmax(-1)
                ce = -(_soft_labels(yc, device) * logp).sum(-1)
                mean = (logp.exp() * model.bin_values).sum(-1)
                l1 = (mean - yc.clamp(0, CAP_S)).abs() / CAP_S
                loss = ((ce + L1_WEIGHT * l1) * wc).sum() / len(idxs)
                if use_stage:
                    # Distil YASA's stage, weighted like the main loss (so
                    # near-onset windows, where N1 matters, count most).
                    stage_ce = -(st[:, s:s + CHUNK]
                                 * stage_logits.log_softmax(-1)).sum(-1)
                    stage_loss = (stage_ce * wc * sm[:, s:s + CHUNK]
                                  ).sum() / len(idxs)
                    loss = loss + STAGE_WEIGHT * stage_loss
                    running_stage += stage_loss.item() * len(idxs)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                running += loss.item() * len(idxs)
            n_rec += len(idxs)
        sched.step()

        probas = [_sequence_proba(model, X_all, idx, device)
                  for idx, _ in val_recs]
        q, score = min(_scores(probas, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        stage_msg = (f" (stage CE {running_stage / n_rec:.3f})"
                     if use_stage else "")
        print(f"[CNN-GRU {tag}] epoch {epoch + 1}/{EPOCHS} "
              f"loss {running / n_rec:.3f}{stage_msg} val W-bMAE {score:.1f} s "
              f"(q={q}) {time.time() - t0:.0f} s")
        if score < best:
            best, bad = score, 0
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return best


def _train_autoencoder(member, X_all, idx, device, tag, max_windows=80000,
                       seed=None):
    """Fit ``member``'s spectral autoencoder on the windows ``idx`` (no
    labels): sets spec_mean / spec_std and the frozen encoder ``ae_enc``."""
    t0 = time.time()
    gen = np.random.default_rng(len(idx))
    idx = gen.choice(idx, min(max_windows, len(idx)), replace=False)
    home = member.spec_mean.device
    member.to(device)
    with torch.no_grad():
        S = torch.cat([member.ae_input(X_all[idx[i:i + 2048]].float()
                                       .to(device))
                       for i in range(0, len(idx), 2048)])
    mean, std = S.mean(0), S.std(0).clamp_min(1e-3)
    S = (S - mean) / std
    n_hold = max(1, len(S) // 20)
    S_val, S_tr = S[:n_hold], S[n_hold:]

    torch.manual_seed(1000 * (MEMBER_SEED if seed is None else seed)
                      + len(idx))
    ae_dim = member.ae_enc[2].out_features
    enc = nn.Sequential(nn.Linear(S.shape[1], AE_HIDDEN), nn.GELU(),
                        nn.Linear(AE_HIDDEN, ae_dim)).to(device)
    dec = nn.Sequential(nn.Linear(ae_dim, AE_HIDDEN), nn.GELU(),
                        nn.Linear(AE_HIDDEN, S.shape[1])).to(device)
    params = list(enc.parameters()) + list(dec.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, AE_EPOCHS)
    for epoch in range(AE_EPOCHS):
        perm = torch.randperm(len(S_tr), device=device)
        total = 0.0
        for i in range(0, len(S_tr), 512):
            x = S_tr[perm[i:i + 512]]
            # Denoising: Gaussian noise and 10 % of the inputs dropped.
            noisy = (x + 0.1 * torch.randn_like(x)) * (
                torch.rand_like(x) > 0.1).float()
            loss = ((dec(enc(noisy)) - x) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(x)
        sched.step()
    with torch.no_grad():
        val = ((dec(enc(S_val)) - S_val) ** 2).mean().item()
    print(f"[CNN-GRU AE {tag}] {len(S_tr)} windows, {S.shape[1]} inputs -> "
          f"{ae_dim}: train MSE {total / len(S_tr):.3f}, held-out MSE "
          f"{val:.3f} (1.0 = predicting the mean) {time.time() - t0:.0f} s")
    member.spec_mean.copy_(mean)
    member.spec_std.copy_(std)
    member.ae_enc.load_state_dict(enc.state_dict())
    member.ae_enc.requires_grad_(False)
    member.to(home)


def _subsample_subjects(recs, frac):
    """Keep the recordings of a random ``frac`` of their subjects."""
    if frac >= 1.0:
        return recs
    subjects = np.unique([s for _, s in recs])
    rng = np.random.default_rng(12345)
    keep = set(rng.choice(subjects, max(1, int(round(frac * len(subjects)))),
                          replace=False))
    return [r for r in recs if r[1] in keep]


def _save_val_predictions(val_recs, rec, sub, y_all, preds):
    """Validation-sleeper predictions -> build/<label>_valpred.npz.

    Training-time analysis only (blend, error per range); written next to
    the exported folder, not inside it, so it never ships in the ZIP.
    """
    sub_dir = os.environ.get("COMPET_SUBMISSION_DIR")
    if not sub_dir:
        return
    sub_dir = os.path.abspath(sub_dir)
    out = os.path.join(os.path.dirname(sub_dir),
                       os.path.basename(sub_dir) + "_valpred.npz")
    idx = np.concatenate([i for i, _ in val_recs])
    # ``record`` = index of the validation recording (identical order in
    # cnn_gru.py and gbm.py), ``record_id`` = the dataset's own id.
    np.savez(out, record=np.concatenate([np.full(len(i), k) for k, (i, _)
                                         in enumerate(val_recs)]),
             record_id=np.asarray(rec[idx]).astype(str),
             subject=np.asarray(sub[idx]).astype(str),
             position=np.concatenate([np.arange(len(i)) for i, _ in val_recs]),
             y=y_all[idx].numpy(), pred=np.concatenate(preds))
    print(f"[CNN-GRU] validation predictions -> {out}")


def member_configs(state):
    """Per-member settings of a saved ensemble state dict.

    Older checkpoints predate some switches: CNN on, feats 1, prep 1, no
    autoencoder, 60 s EMA, no domain flag; the missing buffers are added to
    ``state`` with those values so it loads strictly.
    """
    n = len({k.split(".")[1] for k in state if k.startswith("members.")})
    configs = []
    for i in range(n):
        pre = f"members.{i}."
        for key, val in (("cnn_on", True), ("feats", 1), ("prep_v", 1),
                         ("ae_mode", 0), ("ema_s", 60.0), ("dom_on", False)):
            state.setdefault(pre + key, torch.tensor(val))
        ae = int(state[pre + "ae_mode"])
        configs.append(dict(
            adapt=bool(state[pre + "adapt"]),
            use_cnn=bool(state[pre + "cnn_on"]),
            feats=int(state[pre + "feats"]),
            stage=any(k.startswith(pre + "stage_head.") for k in state),
            prep=int(state[pre + "prep_v"]), ae=ae,
            hidden=int(state[pre + "gru.weight_hh_l0"].shape[1]),
            ae_dim=int(state[pre + "ae_enc.2.weight"].shape[0]) if ae
            else AE_DIM,
            ema_s=float(state[pre + "ema_s"]),
            dom=bool(state[pre + "dom_on"])))
    return configs


def ensemble_from_state(state, n_chans, n_times, sfreq):
    """Saved state dict -> the SleepOnsetEnsemble it came from."""
    configs = member_configs(state)
    q = state["quantile"]
    state.setdefault("member_w", torch.ones(len(configs)))
    state.setdefault("q_time", torch.full((len(TIME_EDGES_MIN) + 1,),
                                          float(q)))
    model = SleepOnsetEnsemble(n_chans, n_times, sfreq, configs=configs)
    model.load_state_dict(state)
    return model


def _load_external(path):
    """scripts/prep_external.py output -> (windows, y, rec, onset, subject).

    Windows are raw microvolts (float16); they get the Muse preprocessing.
    """
    d = np.load(path)
    X = preprocess(torch.from_numpy(d["X"]).float()).half()
    return (X, torch.from_numpy(d["y"]).float(), d["rec"].astype(np.int64),
            d["onset"].astype(np.float64), d["subject"].astype(str),
            d["dataset"].astype(str))


def _save_oof(recs, rec_split, rec_fold, names, y_all, probas, q):
    """Out-of-fold distributions of a CV build -> build/<label>_oof.npz.

    For offline analysis and scripts/merge_builds.py: per window the
    recording index (into the build's recordings, same order in every CV
    build), position, target and the (K,) distribution (float16); per
    recording its split, fold and sleeper. Never shipped in the ZIP.
    """
    sub_dir = os.environ.get("COMPET_SUBMISSION_DIR")
    if not sub_dir:
        return
    sub_dir = os.path.abspath(sub_dir)
    out = os.path.join(os.path.dirname(sub_dir),
                       os.path.basename(sub_dir) + "_oof.npz")
    np.savez_compressed(
        out,
        record=np.concatenate([np.full(len(i), k) for k, (i, _)
                               in enumerate(recs)]),
        position=np.concatenate([np.arange(len(i)) for i, _ in recs]),
        y=np.concatenate([y_all[i].numpy() for i, _ in recs]),
        proba=torch.cat([p.float().cpu() for p in probas]).half().numpy(),
        rec_split=np.asarray(rec_split).astype(str),
        rec_fold=np.asarray(rec_fold),
        rec_subject=np.asarray([names[i[0]] for i, _ in recs]).astype(str),
        quantile=np.float32(q))
    print(f"[CNN-GRU] out-of-fold distributions -> {out}")


class Solver(CompetSolver):

    name = "CNN-GRU-Hazard"

    def load_model(self, meta):
        weights = meta["submission_dir"] / WEIGHTS_FILE
        # Inference (Codabench): load the shipped weights, which fix every
        # member's own settings (a merged ensemble may mix them). A local
        # training run (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            state = torch.load(weights, map_location=meta["device"],
                               weights_only=True)
            model = ensemble_from_state(state, meta["n_chans"],
                                        meta["n_times"], float(meta["sfreq"]))
            return model.to(meta["device"]).eval()
        n_members = CV_FOLDS * CV_SEEDS if CV_FOLDS > 0 else N_SEEDS
        model = SleepOnsetEnsemble(
            meta["n_chans"], meta["n_times"], float(meta["sfreq"]),
            n_members, ADAPT, USE_CNN, FEATS, STAGE_WEIGHT > 0, PREP,
            AE_MODE, dom=bool(EXTERNAL) and CV_FOLDS > 0)
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        if CV_FOLDS > 0:
            return self._fit_cv(model, train_loader)
        rng = np.random.default_rng(SEED)
        X_all, y_all, rec, onset, sub = _load_windows(train_loader.dataset)
        recs = _recordings(y_all, rec, onset, sub, rng)
        use_stage = STAGE_WEIGHT > 0 and hasattr(train_loader.dataset,
                                                 "seg_ds")
        if STAGE_WEIGHT > 0 and not os.path.exists(TEACHER_FILE):
            raise FileNotFoundError(
                f"SLEEP_STAGE_WEIGHT={STAGE_WEIGHT} needs the YASA teacher "
                f"labels at {TEACHER_FILE} (scripts/make_yasa_teacher.py)")
        if use_stage:
            stage_t, stage_m = _teacher_targets(train_loader.dataset, onset)

        # Hold out whole subjects for early stopping and the quantile.
        subjects = np.unique([s for _, s in recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        val_subjects = set(rng.choice(subjects, n_val, replace=False))
        train_recs = [r for r in recs if r[1] not in val_subjects]
        val_recs = [r for r in recs if r[1] in val_subjects]
        train_recs = _subsample_subjects(train_recs, TRAIN_FRAC)

        # Extra training-only recordings; the validation sleepers above are
        # drawn before they are added, so validation stays comparable.
        extra = _extra_windows(train_loader.dataset) if EXTRA_DATA else None
        if extra is not None:
            n0, rec_off = len(y_all), int(rec.max()) + 1
            sub_off = int(np.max(sub)) + 1
            (X_e, y_e, rec_e, onset_e, sub_e), extra_ds = extra
            if use_stage:
                t_e, m_e = _teacher_targets(extra_ds, onset_e)
                stage_t = torch.cat([stage_t, t_e])
                stage_m = torch.cat([stage_m, m_e])
            extra_recs = [(idx + n0, s + sub_off) for idx, s in
                          _recordings(y_e, rec_e, onset_e, sub_e, rng)]
            X_all = torch.cat([X_all, X_e])
            y_all = torch.cat([y_all, y_e])
            rec = np.concatenate([rec, rec_e + rec_off])
            onset = np.concatenate([onset, onset_e])
            sub = np.concatenate([sub, sub_e + sub_off])
            train_recs = train_recs + extra_recs
            recs = recs + extra_recs
            print(f"[CNN-GRU] + {len(extra_recs)} extra training recordings")

        print(f"[CNN-GRU] {len(train_recs)} train / {len(val_recs)} val "
              f"recordings ({len(subjects)} subjects); "
              f"{len(model.members)} members, "
              f"adapt={model.members[0].use_adapt}, "
              f"cnn={model.members[0].use_cnn}, "
              f"feats={model.members[0].n_feats}, extra={extra is not None}, "
              f"train_frac={TRAIN_FRAC}, seed={MEMBER_SEED}, "
              f"stage_weight={STAGE_WEIGHT if use_stage else 0}, "
              f"prep={model.members[0].prep}, ae={model.members[0].ae}")
        if use_stage:
            train_idx = np.concatenate([idx for idx, _ in train_recs])
            m = stage_m[train_idx]
            mean_t = stage_t[train_idx][m].mean(0).tolist()
            print(f"[CNN-GRU] YASA labels for {m.float().mean():.1%} of "
                  f"training windows; mean wake/drowsy/N2+ = "
                  + "/".join(f"{v:.2f}" for v in mean_t))
        weights = {id(idx): _window_weights(y_all[idx].numpy())
                   for idx, _ in recs}

        # Normalisation stats from the training windows, shared by members.
        train_idx = np.concatenate([idx for idx, _ in train_recs])
        sample = X_all[rng.choice(train_idx, min(20000, len(train_idx)),
                                  replace=False)].float()
        chan_std = sample.std(dim=(0, 2)).clamp_min(1e-3)
        first = model.members[0]
        out_device = model.quantile.device
        device = _train_device(str(out_device))
        if first.ae:
            # One autoencoder on the training windows, shared by members.
            _train_autoencoder(first, X_all, train_idx, device, "shared")
            for m in model.members[1:]:
                m.load_state_dict(first.state_dict())
        feats = first.band_powers(sample.to(first.chan_std.device))
        for m in model.members:
            m.chan_std.copy_(chan_std)
            m.bp_mean.copy_(feats.mean(0))
            m.bp_std.copy_(feats.std(0).clamp_min(1e-3))

        model.to(device)
        print(f"[CNN-GRU] training on {device}")

        member_probas = []
        for i, member in enumerate(model.members):
            best = _train_member(member, 1000 * MEMBER_SEED + i, X_all, y_all,
                                 train_recs,
                                 val_recs, weights, device,
                                 f"{i + 1}/{len(model.members)}",
                                 (stage_t, stage_m) if use_stage else None)
            print(f"[CNN-GRU] member {i + 1}: best val W-bMAE {best:.1f} s")
            member_probas.append([_sequence_proba(member, X_all, idx, device)
                                  for idx, _ in val_recs])

        ensemble = [torch.stack(ps).mean(0) for ps in zip(*member_probas)]
        q, score = min(_scores(ensemble, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        model.set_quantile(q)
        model.to(out_device).eval()
        print(f"[CNN-GRU] ensemble val W-bMAE {score:.1f} s, q={q:.2f}")
        _save_val_predictions(
            val_recs, rec, sub, y_all,
            [quantile_seconds(p, q).clamp(0, CAP_S).cpu().numpy()
             for p in ensemble])
        self._check_streaming(model, X_all, val_recs[0][0], out_device)

    def _fit_cv(self, model, train_loader):
        """Cross-validated all-data training (see CV_FOLDS)."""
        rng = np.random.default_rng(SEED)
        train_ds = train_loader.dataset
        X_all, y_all, rec, onset, sub = _load_windows(train_ds)
        n_train = len(y_all)

        # The usual validation recordings (drawn exactly as in ``fit``), only
        # to report a score comparable with the experiments' val W-bMAE.
        base_recs = _recordings(y_all, rec, onset, sub, rng)
        subjects = np.unique([s for _, s in base_recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        classic_val = set(rng.choice(subjects, n_val, replace=False))
        classic_rec = {int(rec[idx[0]]) for idx, s in base_recs
                       if s in classic_val}

        # Sleeper of every window, by name, so a sleeper's recordings from
        # different splits land in the same fold.
        fields = _timeline_fields(train_ds, n_train)
        names = (np.array([f["subject"] for f in fields]) if fields
                 else np.asarray(sub).astype(str))
        use_stage = STAGE_WEIGHT > 0 and fields is not None
        if STAGE_WEIGHT > 0 and not os.path.exists(TEACHER_FILE):
            raise FileNotFoundError(
                f"SLEEP_STAGE_WEIGHT={STAGE_WEIGHT} needs the YASA teacher "
                f"labels at {TEACHER_FILE} (scripts/make_yasa_teacher.py)")
        if use_stage:
            stage_t, stage_m = _teacher_targets(train_ds, onset)

        split_of = np.zeros(n_train, dtype=object)
        split_of[:] = "train"
        wanted = (["val"] if EXTRA_DATA else []) + (["test"] if USE_TEST
                                                     else [])
        for split, ((X_e, y_e, rec_e, onset_e, _), ds) in _load_splits(
                train_ds, wanted).items():
            f_e = _timeline_fields(ds, len(y_e))
            rec_off = int(rec.max()) + 1
            X_all = torch.cat([X_all, X_e])
            y_all = torch.cat([y_all, y_e])
            rec = np.concatenate([rec, rec_e + rec_off])
            onset = np.concatenate([onset, onset_e])
            names = np.concatenate([names, [f["subject"] for f in f_e]])
            split_of = np.concatenate([split_of, np.full(len(y_e), split,
                                                         dtype=object)])
            if use_stage:
                t_e, m_e = _teacher_targets(ds, onset_e)
                stage_t = torch.cat([stage_t, t_e])
                stage_m = torch.cat([stage_m, m_e])

        _, sub_int = np.unique(names, return_inverse=True)
        recs = _recordings(y_all, rec, onset, sub_int, rng)
        rec_split = [split_of[idx[0]] for idx, _ in recs]
        is_classic = [split_of[idx[0]] == "train"
                      and int(rec[idx[0]]) in classic_rec for idx, _ in recs]

        # Folds over sleepers.
        fold_subjects = np.random.default_rng(SEED + 1).permutation(
            np.unique([s for _, s in recs]))
        K = len(model.members) // max(1, CV_SEEDS)
        fold_of = {s: k for k, part in enumerate(np.array_split(
            fold_subjects, K)) for s in part}
        rec_fold = np.array([fold_of[s] for _, s in recs])
        counts = {sp: rec_split.count(sp) for sp in ("train", "val", "test")}
        print(f"[CNN-GRU] CV: {len(recs)} recordings "
              f"({counts['train']} train + {counts['val']} extra + "
              f"{counts['test']} warm-up test) from {len(fold_subjects)} "
              f"sleepers, {K} folds of "
              f"{'/'.join(str((rec_fold == k).sum()) for k in range(K))} "
              f"recordings; cnn={model.members[0].use_cnn}, "
              f"feats={model.members[0].n_feats}, "
              f"stage_weight={STAGE_WEIGHT if use_stage else 0}, "
              f"prep={model.members[0].prep}, ae={model.members[0].ae}")
        if counts["test"]:
            print("[CNN-GRU] note: the warm-up test recordings are in "
                  "training, so this build's test W-bMAE is meaningless")
        weights = {id(idx): _window_weights(y_all[idx].numpy())
                   for idx, _ in recs}

        # Normalisation stats from all windows, shared by members.
        sample = X_all[rng.choice(len(y_all), min(20000, len(y_all)),
                                  replace=False)].float()
        chan_std = sample.std(dim=(0, 2)).clamp_min(1e-3)
        first = model.members[0]
        out_device = model.quantile.device
        device = _train_device(str(out_device))
        if not first.ae:
            feats = first.band_powers(sample.to(first.chan_std.device))
        for m in model.members:
            m.chan_std.copy_(chan_std)
            if not first.ae:
                m.bp_mean.copy_(feats.mean(0))
                m.bp_std.copy_(feats.std(0).clamp_min(1e-3))

        print(f"[CNN-GRU] training on {device}")

        # External pseudo-Muse PSG recordings: training-only, every fold.
        ext_recs, ext_mask = [], None
        if EXTERNAL and model.members[0].dom:
            X_e, y_e, rec_e, onset_e, sub_e, ds_e = _load_external(EXTERNAL)
            n0, rec_off = len(y_all), int(rec.max()) + 1
            _, sub_e_int = np.unique(sub_e, return_inverse=True)
            sub_e_int = sub_e_int + int(sub_int.max()) + 1
            ext_recs = [(idx + n0, s) for idx, s in _recordings(
                y_e, rec_e, onset_e, sub_e_int, rng)]
            X_all = torch.cat([X_all, X_e])
            y_all = torch.cat([y_all, y_e])
            rec = np.concatenate([rec, rec_e + rec_off])
            ext_mask = np.zeros(len(y_all), dtype=bool)
            ext_mask[n0:] = True
            for idx, _ in ext_recs:
                weights[id(idx)] = EXT_WEIGHT * _window_weights(
                    y_all[idx].numpy())
            print(f"[CNN-GRU] + {len(ext_recs)} external recordings "
                  f"({', '.join(f'{n} {c}' for n, c in zip(*np.unique(ds_e[[i[0] - n0 for i, _ in ext_recs]], return_counts=True)))}), "
                  f"{len(y_e)} windows, loss weight x{EXT_WEIGHT}")

        R = len(model.members) // K
        oof_rep = [[None] * len(recs) for _ in range(R)]
        for j, member in enumerate(model.members):
            k, rep_i = j % K, j // K
            tag = f"fold {k + 1}/{K}" + (f" seed {rep_i + 1}/{R}" if R > 1
                                          else "")
            tr = [r for r, f in zip(recs, rec_fold) if f != k] + ext_recs
            va_i = [i for i, f in enumerate(rec_fold) if f == k]
            va = [recs[i] for i in va_i]
            if member.ae:
                # This fold's autoencoder and feature stats, from its
                # training sleepers only (keeps the out-of-fold honest).
                member.to(out_device)
                # Muse windows only: the autoencoder and stats ignore the
                # external recordings.
                tr_idx = np.concatenate([idx for idx, _ in tr
                                         if ext_mask is None
                                         or not ext_mask[idx[0]]])
                _train_autoencoder(member, X_all, tr_idx, device, tag,
                                   seed=MEMBER_SEED + rep_i)
                s = X_all[rng.choice(tr_idx, min(20000, len(tr_idx)),
                                     replace=False)].float()
                with torch.no_grad():
                    f_k = member.band_powers(s.to(member.chan_std.device))
                member.bp_mean.copy_(f_k.mean(0))
                member.bp_std.copy_(f_k.std(0).clamp_min(1e-3))
            member.to(device)
            best = _train_member(member, 1000 * (MEMBER_SEED + rep_i) + k,
                                 X_all, y_all, tr, va, weights, device, tag,
                                 (stage_t, stage_m) if use_stage else None,
                                 ext_mask)
            print(f"[CNN-GRU] {tag.replace('/' + str(K), '', 1)}: {len(tr)} "
                  f"train / {len(va)} held-out recordings, best held-out "
                  f"W-bMAE {best:.1f} s")
            for i in va_i:
                oof_rep[rep_i][i] = _sequence_proba(member, X_all,
                                                    recs[i][0], device)

        if R > 1:
            for r_i, o in enumerate(oof_rep):
                q_r, s_r = min(_scores(o, y_all, recs).items(),
                               key=lambda kv: kv[1])
                print(f"[CNN-GRU] seed {MEMBER_SEED + r_i} alone: out-of-fold "
                      f"W-bMAE {s_r:.1f} s, q={q_r:.2f}")
        oof = [torch.stack(ps).mean(0) for ps in zip(*oof_rep)]

        all_scores = _scores(oof, y_all, recs)
        q, score = min(all_scores.items(), key=lambda kv: kv[1])
        model.set_quantile(q)
        model.to(out_device).eval()

        def subset(mask):
            pairs = [(p, r) for p, r, m in zip(oof, recs, mask) if m]
            if not pairs:
                return float("nan")
            return _scores([p for p, _ in pairs], y_all,
                           [r for _, r in pairs])[q]

        msg = (f"out-of-fold W-bMAE {score:.1f} s on all {len(recs)} "
               f"recordings, q={q:.2f}")
        if counts["test"]:
            msg += (f"; warm-up test recordings "
                    f"{subset([sp == 'test' for sp in rec_split]):.1f} s")
        print(f"[CNN-GRU] {msg}")
        # Comparable with the experiments' val W-bMAE (same 60 recordings,
        # each predicted by the member that did not train on its sleeper).
        print(f"[CNN-GRU] ensemble val W-bMAE {subset(is_classic):.1f} s "
              f"(out-of-fold, the usual validation recordings)")
        _save_oof(recs, rec_split, rec_fold, names, y_all, oof, q)
        classic = [r for r, m in zip(recs, is_classic) if m]
        _save_val_predictions(
            classic, rec, sub_int, y_all,
            [quantile_seconds(p, q).clamp(0, CAP_S).cpu().numpy()
             for p, m in zip(oof, is_classic) if m])
        self._check_streaming(model, X_all, recs[0][0], out_device)

    @staticmethod
    @torch.no_grad()
    def _check_streaming(model, X_all, idx, device):
        """Window-by-window predict must match a whole-sequence forward."""
        idx = idx[:200]
        w = (model.member_w / model.member_w.sum()).to(device)
        p = (torch.stack([_sequence_proba(m, X_all, idx, device)
                          for m in model.members]) * w.view(-1, 1, 1)).sum(0)
        seq = model._seconds(p, 0)
        model.reset_state()
        t0 = time.time()
        stream = torch.cat([model.predict(X_all[i:i + 1].float().to(device))
                            for i in idx])
        per_window = (time.time() - t0) / max(1, len(idx))
        model.reset_state()
        diff = (seq.clamp(0, CAP_S) - stream).abs().max().item()
        print(f"[CNN-GRU] streaming check: max |seq - stream| = {diff:.4f} s; "
              f"{len(model.members)} members, {1000 * per_window:.1f} ms per "
              f"window on {device}")
        if diff > BIN_S:
            raise RuntimeError("streamed predictions differ from the "
                               "sequence forward pass")

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile submission/gbm.py
"""Track 03 (Sleep Onset) submission: classical ML baseline.

Hand-crafted spectral features per 5 s window + causal context features over
the recording so far -> scikit-learn HistGradientBoostingRegressor trained
with an absolute-error loss and W-bMAE-like sample weights.

Per-window features (per channel unless noted): log absolute and relative
band powers (delta, theta, alpha, sigma, beta, high/EMG), theta/alpha ratio,
spectral entropy, 95 % spectral edge, log variance, and one AF7-AF8 slow
(< 1 Hz) power as an eye-movement proxy.

Context features, computed causally from the windows already seen in the
recording (the streaming state, cleared by ``reset_state``): log elapsed
time, the expanding mean of each feature and the current window's deviation
from it, ~1 min and ~5 min EMAs, and the change since the first 2 minutes.

``SLEEP_GBM_FEATURES=time`` keeps only the elapsed time (a control that
shows how much the EEG adds). ``SLEEP_TRAIN_FRAC`` trains on a fraction of
the training sleepers. The validation sleepers are drawn exactly as in
``cnn_gru.py``, so validation scores are comparable across the two solvers.

Self-contained: numpy, scikit-learn, joblib and torch only.
"""

import os
import time

import joblib
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

from benchmark_utils.base_solver import CompetSolver
from benchmark_utils.data import to_numpy

CAP_S = 600.0
WEIGHTS_FILE = "weights.joblib"
CLIP_UV = 500.0
BANDS = ((0.5, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 15.0), (15.0, 30.0),
         (30.0, 60.0))
RANGE_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
RANGE_WEIGHTS = (10.0, 5.0, 3.0, 1.0)

FEATURES = os.environ.get("SLEEP_GBM_FEATURES", "all")      # all | time
TRAIN_FRAC = float(os.environ.get("SLEEP_TRAIN_FRAC", 1.0))
MEMBER_SEED = int(os.environ.get("SLEEP_SEED", 0))
SEED = 0                                 # validation split seed (fixed)
VAL_FRACTION = 0.15                      # same as cnn_gru.py

EMA_FAST_S = 60.0
EMA_SLOW_S = 300.0
BASELINE_S = 120.0

GBM_PARAMS = dict(loss="absolute_error", learning_rate=0.05, max_iter=600,
                  max_leaf_nodes=31, min_samples_leaf=200,
                  l2_regularization=1.0, max_features=0.5)


# --- features ----------------------------------------------------------------


def window_features(X, sfreq):
    """Raw windows (N, C, T) microvolts -> per-window features (N, F)."""
    X = np.asarray(X, dtype=np.float64)
    X = X - X.mean(axis=-1, keepdims=True)
    X = np.clip(X, -CLIP_UV, CLIP_UV)
    N, C, T = X.shape
    freqs = np.fft.rfftfreq(T, d=1.0 / sfreq)
    spec = np.abs(np.fft.rfft(X * np.hanning(T), axis=-1)) ** 2   # (N, C, K)

    nyq = sfreq / 2
    power = np.stack([spec[..., (freqs >= lo) & (freqs < min(hi, nyq))]
                      .sum(-1) for lo, hi in BANDS], axis=-1)       # (N, C, B)
    log_abs = np.log(power + 1e-6)
    log_rel = np.log(power / (power.sum(-1, keepdims=True) + 1e-6) + 1e-6)
    theta_alpha = log_abs[..., 1] - log_abs[..., 2]

    band = (freqs >= 0.5) & (freqs < min(60.0, nyq))
    p = spec[..., band]
    p = p / (p.sum(-1, keepdims=True) + 1e-12)
    entropy = -(p * np.log(p + 1e-12)).sum(-1) / np.log(p.shape[-1])
    edge = freqs[band][np.argmax(p.cumsum(-1) >= 0.95, axis=-1)]
    log_var = np.log(X.var(-1) + 1e-6)

    a, b = (1, 2) if C >= 3 else (0, min(1, C - 1))   # AF7, AF8 on Muse
    eog = X[:, a] - X[:, b]
    eog_spec = np.abs(np.fft.rfft(eog * np.hanning(T), axis=-1)) ** 2
    slow = np.log(eog_spec[:, (freqs > 0) & (freqs < 1.0)].sum(-1) + 1e-6)

    return np.concatenate([
        log_abs.reshape(N, -1), log_rel.reshape(N, -1), theta_alpha, entropy,
        edge, log_var, slow[:, None]], axis=1).astype(np.float32)


class ContextState:
    """Causal per-recording summaries of the window features."""

    def __init__(self, window_s):
        self.window_s = window_s
        self.a_fast = min(1.0, window_s / EMA_FAST_S)
        self.a_slow = min(1.0, window_s / EMA_SLOW_S)
        self.n_base = max(1, int(round(BASELINE_S / window_s)))
        self.reset()

    def reset(self):
        self.n = 0
        self.total = self.fast = self.slow = self.base_sum = None

    def step(self, f):
        """Features of the next window (F,) -> context features (D,)."""
        f = f.astype(np.float64)
        if self.n == 0:
            self.total = np.zeros_like(f)
            self.base_sum = np.zeros_like(f)
            self.fast = f.copy()
            self.slow = f.copy()
        else:
            self.fast += self.a_fast * (f - self.fast)
            self.slow += self.a_slow * (f - self.slow)
        self.n += 1
        self.total += f
        if self.n <= self.n_base:
            self.base_sum += f
        mean = self.total / self.n
        base = self.base_sum / min(self.n, self.n_base)
        elapsed_min = (self.n - 1) * self.window_s / 60.0
        t = np.array([np.log1p(elapsed_min), elapsed_min])
        if FEATURES == "time":
            return t
        return np.concatenate([t, f, f - mean, self.fast, self.slow,
                               self.fast - base, self.slow - base,
                               self.fast - self.slow])

    def run(self, feats):
        """Features (L, F) of consecutive windows -> context (L, D)."""
        return np.stack([self.step(f) for f in feats]).astype(np.float32)


# --- model -------------------------------------------------------------------


class SleepOnsetGBM:
    """Streaming wrapper: window features -> context -> gradient boosting."""

    def __init__(self, sfreq, n_times, estimator=None):
        self.sfreq = float(sfreq)
        self.window_s = n_times / self.sfreq
        self.clf = estimator
        self.state = ContextState(self.window_s)

    def reset_state(self):
        self.state.reset()

    def predict(self, X):
        """Consecutive windows (B, C, T) of one recording -> (B,) seconds."""
        ctx = self.state.run(window_features(to_numpy(X), self.sfreq))
        if self.clf is None:
            return np.full(len(ctx), CAP_S / 2)
        return np.clip(self.clf.predict(ctx), 0.0, CAP_S)


# --- training helpers (local only) -------------------------------------------


def _window_weights(y):
    """Per-window weights of one recording, mimicking its W-bMAE (sum 1)."""
    edges = np.asarray(RANGE_EDGES)
    r = np.clip(np.digitize(y, edges[1:-1]), 0, len(RANGE_WEIGHTS) - 1)
    w = np.zeros(len(y))
    present = np.unique(r)
    total = sum(RANGE_WEIGHTS[b] for b in present)
    for b in present:
        mask = r == b
        w[mask] = RANGE_WEIGHTS[b] / total / mask.sum()
    return w


def _load(dataset, sfreq):
    """Every window once -> features (N, F), y, record, onset, subject."""
    import torch
    from torch.utils.data import DataLoader
    F, ys, recs, onsets, subs = [], [], [], [], []
    t0 = time.time()
    for X, y, info in DataLoader(dataset, batch_size=512, shuffle=False):
        F.append(window_features(X.float().cpu().numpy(), sfreq))
        ys.append(torch.as_tensor(y).float().cpu().reshape(-1).numpy())
        recs.append(np.asarray(info["record_id"]).reshape(-1))
        onsets.append(np.asarray(info["onset"], dtype=np.float64).reshape(-1))
        subs.append(np.asarray(info.get("subject_id", info["record_id"]))
                    .reshape(-1))
    print(f"[GBM] features of {sum(len(f) for f in F)} windows in "
          f"{time.time() - t0:.0f} s")
    return (np.concatenate(F), np.concatenate(ys), np.concatenate(recs),
            np.concatenate(onsets), np.concatenate(subs))


def _recordings(y, rec, onset, sub, rng):
    """Same as cnn_gru._recordings: time-ordered indices + subject."""
    out = []
    if len(np.unique(rec)) < 4:
        perm = rng.permutation(len(y))
        for i, part in enumerate(np.array_split(perm, 10)):
            out.append((part[np.argsort(-y[part], kind="stable")], i))
        return out
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        out.append((idx[np.argsort(onset[idx], kind="stable")],
                    int(sub[idx[0]])))
    return out


def _subsample_subjects(recs, frac):
    """Same as cnn_gru._subsample_subjects."""
    if frac >= 1.0:
        return recs
    subjects = np.unique([s for _, s in recs])
    rng = np.random.default_rng(12345)
    keep = set(rng.choice(subjects, max(1, int(round(frac * len(subjects)))),
                          replace=False))
    return [r for r in recs if r[1] in keep]


def _stack(recs, feats, y_all, window_s):
    X, y, w = [], [], []
    for idx, _ in recs:
        X.append(ContextState(window_s).run(feats[idx]))
        y.append(np.clip(y_all[idx], 0, CAP_S))
        w.append(_window_weights(y_all[idx]))
    return np.concatenate(X), np.concatenate(y), np.concatenate(w)


def _wbmae(clf, recs, feats, y_all, window_s):
    from benchmark_utils.metrics import binned_mae
    scores, preds = [], []
    for idx, _ in recs:
        p = np.clip(clf.predict(ContextState(window_s).run(feats[idx])),
                    0, CAP_S)
        preds.append(p)
        scores.append(binned_mae(y_all[idx], p, RANGE_EDGES, RANGE_WEIGHTS))
    return float(np.mean(scores)), preds


def _save_val_predictions(val_recs, rec, sub, y_all, preds):
    """Validation predictions -> build/<label>_valpred.npz (not zipped)."""
    sub_dir = os.environ.get("COMPET_SUBMISSION_DIR")
    if not sub_dir:
        return
    sub_dir = os.path.abspath(sub_dir)
    out = os.path.join(os.path.dirname(sub_dir),
                       os.path.basename(sub_dir) + "_valpred.npz")
    idx = np.concatenate([i for i, _ in val_recs])
    # ``record`` = index of the validation recording (identical order in
    # cnn_gru.py and gbm.py), ``record_id`` = the dataset's own id.
    np.savez(out, record=np.concatenate([np.full(len(i), k) for k, (i, _)
                                         in enumerate(val_recs)]),
             record_id=np.asarray(rec[idx]).astype(str),
             subject=np.asarray(sub[idx]).astype(str),
             position=np.concatenate([np.arange(len(i)) for i, _ in val_recs]),
             y=y_all[idx], pred=np.concatenate(preds))
    print(f"[GBM] validation predictions -> {out}")


class Solver(CompetSolver):

    name = "GBM-Context"

    def load_model(self, meta):
        weights = meta["submission_dir"] / WEIGHTS_FILE
        estimator = None
        if weights.exists() and self.train_loader is None:
            estimator = joblib.load(weights)
        return SleepOnsetGBM(meta["sfreq"], meta["n_times"], estimator)

    def fit(self, model, train_loader):
        rng = np.random.default_rng(SEED)
        feats, y_all, rec, onset, sub = _load(train_loader.dataset,
                                              model.sfreq)
        recs = _recordings(y_all, rec, onset, sub, rng)
        subjects = np.unique([s for _, s in recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        val_subjects = set(rng.choice(subjects, n_val, replace=False))
        train_recs = [r for r in recs if r[1] not in val_subjects]
        val_recs = [r for r in recs if r[1] in val_subjects]
        train_recs = _subsample_subjects(train_recs, TRAIN_FRAC)
        print(f"[GBM] {len(train_recs)} train / {len(val_recs)} val "
              f"recordings ({len(subjects)} subjects); features={FEATURES}, "
              f"train_frac={TRAIN_FRAC}, seed={MEMBER_SEED}")

        Xtr, ytr, wtr = _stack(train_recs, feats, y_all, model.window_s)
        print(f"[GBM] training on {Xtr.shape[0]} windows x "
              f"{Xtr.shape[1]} features")

        # Pick the number of boosting rounds on the validation sleepers'
        # W-bMAE, by growing the ensemble with warm_start.
        params = {**GBM_PARAMS, "min_samples_leaf": max(
            5, min(GBM_PARAMS["min_samples_leaf"], len(ytr) // 50))}
        clf = HistGradientBoostingRegressor(
            **params, early_stopping=False, warm_start=True,
            random_state=MEMBER_SEED)
        best, best_iter, t0 = float("inf"), None, time.time()
        for n_iter in range(50, GBM_PARAMS["max_iter"] + 1, 50):
            clf.set_params(max_iter=n_iter)
            clf.fit(Xtr, ytr, sample_weight=wtr)
            score, _ = _wbmae(clf, val_recs, feats, y_all, model.window_s)
            print(f"[GBM] {n_iter} trees: val W-bMAE {score:.1f} s "
                  f"({time.time() - t0:.0f} s)")
            if score < best:
                best, best_iter = score, n_iter

        clf = HistGradientBoostingRegressor(
            **{**params, "max_iter": best_iter}, early_stopping=False,
            random_state=MEMBER_SEED)
        clf.fit(Xtr, ytr, sample_weight=wtr)
        score, preds = _wbmae(clf, val_recs, feats, y_all, model.window_s)
        model.clf = clf
        print(f"[GBM] ensemble val W-bMAE {score:.1f} s, {best_iter} trees")
        _save_val_predictions(val_recs, rec, sub, y_all, preds)

    def save_model(self, model, path):
        joblib.dump(model.clf, path / WEIGHTS_FILE)

In [ ]:
%%writefile submission/submission.py
"""Track 03 (Sleep Onset) submission: a basic MLP regressor.

Codabench only calls ``Solver.load_model(meta)`` and the returned model's
``predict(X)``; ``fit`` and ``save_model`` are local-training hooks used by
``benchopt run ... -o "Sleep-onset[training=True]"`` to produce ``weights.pt``.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

HIDDEN = (256, 64)
DROPOUT = 0.2
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4


class SleepOnsetMLP(nn.Module):
    """(B, C, T) -> per-channel standardisation -> flatten -> MLP -> (B,).

    The per-channel mean/std are fitted on the training set and stored as
    buffers, so they ship inside ``weights.pt`` with the rest of the model.
    """

    def __init__(self, n_chans, n_times, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.register_buffer("chan_mean", torch.zeros(1, n_chans, 1))
        self.register_buffer("chan_std", torch.ones(1, n_chans, 1))
        layers, in_dim = [], n_chans * n_times
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, X):
        X = (X - self.chan_mean) / self.chan_std
        # Fraction of CAP_S in [0, 1].
        return torch.sigmoid(self.net(X.flatten(1))).squeeze(-1)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        return self(X.float()) * CAP_S  # (B,) seconds

    def reset_state(self):
        # Stateless: nothing to reset between streamed recordings.
        pass


class Solver(CompetSolver):

    name = "Sleep-MLP"

    def load_model(self, meta):
        model = SleepOnsetMLP(meta["n_chans"], meta["n_times"])
        weights = meta["submission_dir"] / WEIGHTS_FILE
        # Inference (Codabench): load the shipped weights. A local training
        # run (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            model.load_state_dict(torch.load(
                weights, map_location=meta["device"], weights_only=True))
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        # Per-channel normalisation stats over the whole training set.
        total, total_sq, count = 0.0, 0.0, 0
        for X, _y, _info in train_loader:
            X = X.float()
            total = total + X.sum(dim=(0, 2))
            total_sq = total_sq + (X ** 2).sum(dim=(0, 2))
            count += X.shape[0] * X.shape[2]
        mean = total / count
        std = (total_sq / count - mean ** 2).clamp_min(1e-12).sqrt()
        model.chan_mean.copy_(mean.view(1, -1, 1))
        model.chan_std.copy_(std.view(1, -1, 1))

        opt = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        # L1 matches the MAE-based track metric.
        loss_fn = nn.L1Loss()
        model.train()
        for epoch in range(EPOCHS):
            running, n = 0.0, 0
            for X, y, _info in train_loader:
                opt.zero_grad()
                loss = loss_fn(model(X.float()), y.float().clamp(0, CAP_S) / CAP_S)
                loss.backward()
                opt.step()
                running += loss.item() * len(y)
                n += len(y)
            print(f"[Sleep-MLP] epoch {epoch + 1}/{EPOCHS} "
                  f"train MAE {running / n * CAP_S:.1f} s")
        model.eval()

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile scripts/build_submission.py
#!/usr/bin/env python3
"""Train submission/submission.py with Benchopt, re-validate it, and ZIP it.

Steps (mirrors competition/tools/debug_submission.py):
1. Training run (``Sleep-onset[training=True]``): Solver.fit trains the model
   and the export writes submission.py + weights.pt into build/<label>/.
2. Inference-only run on the exported folder, exactly as Codabench runs it,
   to catch architecture/checkpoint mismatches.
3. ZIP the folder's files at the archive root into dist/<label>.zip.

Usage:
    python scripts/build_submission.py                 # synthetic, warm-up shapes
    python scripts/build_submission.py --data muse     # public Muse warm-up data
    python scripts/build_submission.py --data muse --solver submission/my_cnn.py
"""

from __future__ import annotations

import argparse
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
TRACK = ROOT / "competition" / "tracks" / "sleep_onset"
DEFAULT_SOLVER = ROOT / "submission" / "submission.py"

DATASETS = {
    # Synthetic data with the real warm-up tensor contract (4 Muse channels,
    # 5 s at 128 Hz). Validates the pipeline only; the score is meaningless.
    "simulated": "Simulated[n_chans=4,n_times=640,n_train=300,n_test=150,sfreq=128]",
    # Public Muse warm-up data; run `benchopt prepare` on it first.
    "muse": "Interaxon2026Muse",
}


def run(cmd: list[str], env: dict[str, str]) -> None:
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=ROOT)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", choices=DATASETS, default="simulated")
    parser.add_argument("--solver", type=Path, default=DEFAULT_SOLVER,
                        help="solver file to train (default: submission/submission.py)")
    parser.add_argument("--label",
                        help="output name (default: <solver file stem>_<data>)")
    parser.add_argument("--force", action="store_true",
                        help="overwrite an existing build/dist output")
    args = parser.parse_args()

    benchopt = shutil.which("benchopt") or str(Path(sys.executable).with_name("benchopt"))
    if not Path(benchopt).exists():
        raise SystemExit("benchopt not found; activate the sleep_foundation env")

    solver = args.solver.resolve()
    if not solver.is_file():
        raise SystemExit(f"solver file not found: {solver}")
    stem = "mlp" if solver == DEFAULT_SOLVER else solver.stem
    label = args.label or f"{stem}_{args.data}"
    build_dir = ROOT / "build" / label
    zip_path = ROOT / "dist" / f"{label}.zip"
    if build_dir.exists() or zip_path.exists():
        if not args.force:
            raise SystemExit(f"{build_dir} or {zip_path} exists; use --force")
        shutil.rmtree(build_dir, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
    build_dir.mkdir(parents=True)
    zip_path.parent.mkdir(exist_ok=True)

    env = {**os.environ, "COMPET_SUBMISSION_DIR": str(build_dir)}
    common = [benchopt, "run", str(TRACK), "-d", DATASETS[args.data],
              "-r", "1", "--no-cache", "--no-plot", "--no-html",
              "--no-display", "--local"]

    # 1. Train and export.
    run(common + ["-o", "Sleep-onset[training=True]", "-s", str(solver),
                  "--output", str(ROOT / "build" / f"{label}_train.parquet")], env)
    if not (build_dir / "submission.py").is_file():
        raise SystemExit("training run did not export submission.py")
    if not any(p.is_file() and p.name != "submission.py"
               for p in build_dir.iterdir()):
        raise SystemExit("training run did not export any weights file")

    # 2. Reload the exported folder inference-only, as Codabench does.
    run(common + ["-o", "Sleep-onset[training=False]",
                  "-s", str(build_dir / "submission.py"),
                  "--output", str(ROOT / "build" / f"{label}_eval.parquet")], env)

    # 3. ZIP with every file at the archive root.
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(build_dir.iterdir()):
            if path.is_file():
                zf.write(path, path.name)
    print(f"\nReady to upload: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            print(f"  {info.filename:20s} {info.file_size:>12,d} bytes")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/merge_builds.py
#!/usr/bin/env python3
"""Merge cross-validated CNN-GRU builds into one ensemble submission.

Every CV build (SLEEP_CV_FOLDS > 0) writes build/<label>_oof.npz: the
out-of-fold time-to-N2 distribution of every window of the 540 public
recordings. All CV builds use the same folds and recording order, so their
out-of-fold distributions can be mixed honestly: each recording is still
predicted only by models that never saw its sleeper.

1. Greedy ensemble selection with replacement (Caruana et al. 2004) picks
   build weights that minimise the out-of-fold W-bMAE, and coordinate descent
   picks the quantile per elapsed-time bucket (``q_time``). To show how much
   of the gain is selection luck, both are first fitted on folds 1-4 and
   scored on fold 5 (honest), then refitted on all folds.
2. The selected builds' members are concatenated into one SleepOnsetEnsemble
   (members of a build share its weight), saved as build/<label>/weights.pt
   with the solver as submission.py.
3. Unless --no-eval, the folder is reloaded inference-only with benchopt on
   the Muse data (as Codabench does, like scripts/build_submission.py), then
   zipped into dist/<label>.zip.

Usage:
    python scripts/merge_builds.py --label r8_merged r8_combo_s0 r8_ae64_s0 ...
"""

import argparse
import importlib.util
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
TRACK = ROOT / "competition" / "tracks" / "sleep_onset"
SOLVER = ROOT / "submission" / "cnn_gru.py"
EDGES = np.array([0.0, 40.0, 90.0, 300.0, 600.0])
WEIGHTS = np.array([10.0, 5.0, 3.0, 1.0])
QUANTILES = (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65)
TIME_EDGES_MIN = (2.0, 5.0, 10.0, 20.0)
BIN_S = 5.0


def load_oof(label):
    d = np.load(ROOT / "build" / f"{label}_oof.npz")
    return {k: d[k] for k in d.files}


class Scorer:
    """W-bMAE per recording (benchmark_utils.metrics.binned_mae), averaged."""

    def __init__(self, oof, window_s=5.0):
        self.rec = oof["record"]
        self.y = oof["y"].astype(np.float64)
        self.n_rec = int(self.rec.max()) + 1
        keep = (self.y >= EDGES[0]) & (self.y <= EDGES[-1])
        rng = np.clip(np.digitize(self.y, EDGES[1:-1]), 0, 3)
        self.cell = np.where(keep, self.rec * 4 + rng, -1)
        counts = np.bincount(self.cell[keep], minlength=self.n_rec * 4)
        self.counts = counts.reshape(self.n_rec, 4)
        present = self.counts > 0
        self.norm = (present * WEIGHTS).sum(1)
        self.present = present
        self.keep = keep
        minutes = oof["position"] * window_s / 60.0
        self.bucket = np.digitize(minutes, TIME_EDGES_MIN, right=False)

    def per_recording(self, pred):
        err = np.abs(pred - self.y)
        sums = np.bincount(self.cell[self.keep], weights=err[self.keep],
                           minlength=self.n_rec * 4).reshape(self.n_rec, 4)
        means = np.where(self.present, sums / np.maximum(self.counts, 1), 0)
        return (means * WEIGHTS).sum(1) / np.maximum(self.norm, 1e-9)

    def score(self, pred, recs):
        return float(self.per_recording(pred)[recs].mean())


def quantile_pred(cdf, q_per_window):
    k = (cdf < q_per_window[:, None]).sum(1)
    return np.minimum(k, cdf.shape[1] - 1) * BIN_S


def best_q(scorer, cdf, recs):
    """Best single quantile -> (score, q)."""
    return min((scorer.score(quantile_pred(cdf, np.full(len(cdf), q)), recs),
                q) for q in QUANTILES)


def tune_q_time(scorer, cdf, recs, q0):
    """Coordinate descent over the elapsed-time buckets' quantiles."""
    q_time = np.full(len(TIME_EDGES_MIN) + 1, q0)
    best = scorer.score(quantile_pred(cdf, q_time[scorer.bucket]), recs)
    for _ in range(2):
        for b in range(len(q_time)):
            for q in QUANTILES:
                trial = q_time.copy()
                trial[b] = q
                s = scorer.score(quantile_pred(cdf, trial[scorer.bucket]),
                                 recs)
                if s < best - 1e-6:
                    best, q_time = s, trial
    return best, q_time


def greedy(scorer, probas, recs, steps):
    """Caruana selection with replacement -> (counts per build, score, q)."""
    counts = np.zeros(len(probas), dtype=int)
    total = np.zeros_like(probas[0], dtype=np.float32)
    best_score, best_q_ = np.inf, 0.5
    for _ in range(steps):
        trials = []
        for b, p in enumerate(probas):
            mix = (total + p) / (counts.sum() + 1)
            s, q = best_q(scorer, np.cumsum(mix, 1), recs)
            trials.append((s, q, b))
        s, q, b = min(trials)
        if s >= best_score - 0.01:
            break
        best_score, best_q_ = s, q
        counts[b] += 1
        total += probas[b]
    return counts, best_score, best_q_


def mix_cdf(probas, counts):
    total = sum(c * p for c, p in zip(counts, probas) if c)
    return np.cumsum(total / counts.sum(), 1)


def load_solver():
    sys.path.insert(0, str(TRACK))
    spec = importlib.util.spec_from_file_location("cnn_gru", SOLVER)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def merge_states(labels, counts, q_time, mod):
    import torch
    merged, weights, j = {}, [], 0
    for label, c in zip(labels, counts):
        if not c:
            continue
        state = torch.load(ROOT / "build" / label / "weights.pt",
                           map_location="cpu", weights_only=True)
        mod.member_configs(state)            # fills old checkpoints' buffers
        n = len({k.split(".")[1] for k in state if k.startswith("members.")})
        w_old = state.get("member_w", torch.ones(n)).float()
        for i in range(n):
            pre = f"members.{i}."
            for k, v in state.items():
                if k.startswith(pre):
                    merged[f"members.{j}." + k[len(pre):]] = v
            weights.append(float(c) * float(w_old[i] / w_old.sum()))
            j += 1
    merged["member_w"] = torch.tensor(weights)
    merged["q_time"] = torch.tensor(q_time, dtype=torch.float32)
    merged["quantile"] = torch.tensor(float(np.median(q_time)))
    return merged


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("builds", nargs="+", help="CV build labels (build/<label>)")
    ap.add_argument("--label", required=True, help="output build name")
    ap.add_argument("--steps", type=int, default=20)
    ap.add_argument("--no-eval", action="store_true",
                    help="skip the benchopt inference-only check (no Muse data)")
    args = ap.parse_args()

    labels = [b for b in args.builds
              if (ROOT / "build" / f"{b}_oof.npz").exists()
              and (ROOT / "build" / b / "weights.pt").exists()]
    missing = sorted(set(args.builds) - set(labels))
    if missing:
        print(f"skipping (no _oof.npz or weights): {', '.join(missing)}")
    if not labels:
        raise SystemExit("no usable builds")
    oofs = [load_oof(b) for b in labels]
    for b, o in zip(labels[1:], oofs[1:]):
        for key in ("record", "position", "y"):
            if not np.array_equal(o[key], oofs[0][key]):
                raise SystemExit(f"{b}: recordings differ from {labels[0]} "
                                 f"({key}); not the same CV setup")
    scorer = Scorer(oofs[0])
    probas = [o["proba"].astype(np.float32) for o in oofs]
    rec_fold = oofs[0]["rec_fold"]
    all_recs = np.arange(scorer.n_rec)
    print(f"{len(labels)} builds, {scorer.n_rec} recordings, "
          f"{len(scorer.y)} windows")

    print("\nsingle builds (out-of-fold W-bMAE, all recordings):")
    singles = []
    for b, p in zip(labels, probas):
        s, q = best_q(scorer, np.cumsum(p, 1), all_recs)
        singles.append(s)
        print(f"  {b:28s} {s:6.2f} s  (q={q})")

    # Honest check: select and tune on folds 1-4, score fold 5.
    last = int(rec_fold.max())
    fit_r, hold_r = all_recs[rec_fold != last], all_recs[rec_fold == last]
    counts, _, q = greedy(scorer, probas, fit_r, args.steps)
    _, q_time = tune_q_time(scorer, mix_cdf(probas, counts), fit_r, q)
    cdf = mix_cdf(probas, counts)
    hold_ens = scorer.score(quantile_pred(cdf, q_time[scorer.bucket]), hold_r)
    hold_single = min(
        (best_q(scorer, np.cumsum(p, 1), fit_r)[0],
         scorer.score(quantile_pred(np.cumsum(p, 1), np.full(
             len(p), best_q(scorer, np.cumsum(p, 1), fit_r)[1])), hold_r))
        for p in probas)[1]
    print(f"\nhonest check, fitted on folds 1-{last}, scored on fold "
          f"{last + 1}: ensemble {hold_ens:.2f} s vs best single build "
          f"{hold_single:.2f} s")

    # Final: fit on all recordings.
    counts, s_sel, q = greedy(scorer, probas, all_recs, args.steps)
    s_time, q_time = tune_q_time(scorer, mix_cdf(probas, counts), all_recs, q)
    print("\nselected (weight = times picked):")
    for b, c in zip(labels, counts):
        if c:
            print(f"  {b:28s} x{c}")
    print(f"out-of-fold W-bMAE: best single {min(singles):.2f} s, "
          f"ensemble {s_sel:.2f} s (q={q}), + time quantiles {s_time:.2f} s "
          f"(q_time={q_time.tolist()} for <{TIME_EDGES_MIN} min)")

    mod = load_solver()
    import torch
    state = merge_states(labels, counts, q_time, mod)
    model = mod.ensemble_from_state(dict(state), 4, 640, 128.0)
    out_dir = ROOT / "build" / args.label
    zip_path = ROOT / "dist" / f"{args.label}.zip"
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    torch.save(model.state_dict(), out_dir / "weights.pt")
    shutil.copy(SOLVER, out_dir / "submission.py")
    print(f"\n{len(model.members)} members -> {out_dir}")

    if not args.no_eval:
        benchopt = shutil.which("benchopt") or str(
            Path(sys.executable).with_name("benchopt"))
        cmd = [benchopt, "run", str(TRACK), "-d", "Interaxon2026Muse",
               "-r", "1", "--no-cache", "--no-plot", "--no-html",
               "--no-display", "--local", "-o", "Sleep-onset[training=False]",
               "-s", str(out_dir / "submission.py"),
               "--output", str(ROOT / "build" / f"{args.label}_eval.parquet")]
        print("+", " ".join(cmd), flush=True)
        subprocess.run(cmd, check=True, cwd=ROOT,
                       env={**os.environ, "COMPET_SUBMISSION_DIR": str(out_dir)})

    zip_path.parent.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(out_dir.iterdir()):
            if path.is_file():
                zf.write(path, path.name)
    print(f"\nReady to upload: {zip_path}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/prep_external.py
#!/usr/bin/env python3
"""External PSG sleep onsets as pseudo-Muse training windows.

Downloads two open PhysioNet PSG datasets from the anonymous S3 mirror
(``physionet-open``), one recording at a time (download, cut, delete, so the
disk never holds more than a few recordings), and writes their pre-N2
windows in the Muse task's format to one compressed file:

* Sleep-EDF Expanded, sleep cassette (Kemp et al. 2000; 153 nights, 78
  sleepers, 100 Hz; ~7.6 GB): Fpz-Cz, Pz-Oz, horizontal EOG. R&K stage 2 is
  taken as N2.
* HMC sleep staging (Alvarez-Estevez & Rijsman 2021/2022; 151 nights, 256 Hz;
  ~17 GB): F4-M1, C3-M2, E1-M2, E2-M2 (plus C4, O2), AASM stages.

Pseudo-Muse channels (Muse records TP9, AF7, AF8, TP10 against Fpz; a
mastoid is close to TP9/TP10, and Fpz to the frontal sites), in microvolts:

    HMC        TP9 = -(F4-M1)   AF7 = (E1-M2) - (F4-M1)
               TP10 = -(C3-M2)  AF8 = (E2-M2) - (F4-M1)   (AF7-AF8 = E1-E2)
    Sleep-EDF  TP9 = -(Fpz-Cz)  AF7 = +EOG / 2
               TP10 = Pz-Oz     AF8 = -EOG / 2            (AF7-AF8 = EOG)

These only approximate Muse; the solver gives them a domain flag and a lower
loss weight, and never scores them.

Each stream starts U seconds before the first N2 epoch. The Muse recordings
start when the sleeper starts the recording, so U is HMC's own lights-off
time when it is annotated (and 60-1800 s before N2); otherwise U is drawn
from the public Muse onset latencies (MUSE_ONSET_PCT), so the elapsed-time
input means the same as on Muse. As in the Muse task, the stream is cut into
5 s windows at 128 Hz from its start; a window ending at time t has target
``min(first_N2 - t, 600)`` s.

Output (``--out``, default external_windows.npz): X (N, 4, 640) float16 raw
microvolts; y float32; rec int (recording index); onset float (window start
within its stream, s); subject str; dataset str.

Usage:
    python scripts/prep_external.py                        # both, all nights
    python scripts/prep_external.py --limit 2 --out /tmp/ext_test.npz
"""

import argparse
import concurrent.futures as cf
import os
import tempfile
import time
from pathlib import Path

import numpy as np

BUCKET = "physionet-open"
SFREQ = 128.0
WIN_S = 5.0
CAP_S = 600.0
MAX_U_S = 1800.0
# Percentiles 0, 5, ..., 100 of the 540 public Muse onset latencies (s).
MUSE_ONSET_PCT = [60, 268, 330, 390, 420, 450, 510, 540, 600, 630, 690, 750,
                  810, 870, 960, 1001, 1080, 1197, 1270, 1378, 1500]
T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.0f} s] {msg}", flush=True)


def s3():
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    return boto3.client("s3", config=Config(signature_version=UNSIGNED,
                                            max_pool_connections=32))


def list_keys(client, prefix):
    keys = []
    for page in client.get_paginator("list_objects_v2").paginate(
            Bucket=BUCKET, Prefix=prefix):
        keys += [o["Key"] for o in page.get("Contents", [])]
    return keys


def recordings(client, dataset):
    """[(subject, psg key, annotation key)] of one dataset."""
    if dataset == "sleepedf":
        keys = list_keys(client, "sleep-edfx/1.0.0/sleep-cassette/")
        psg = {Path(k).name[:6]: k for k in keys if k.endswith("-PSG.edf")}
        hyp = {Path(k).name[:6]: k for k in keys
               if k.endswith("-Hypnogram.edf")}
        # SC4ssN: ss = subject, N = night.
        return [(f"sleepedf-{n[3:5]}", psg[n], hyp[n])
                for n in sorted(psg) if n in hyp]
    keys = list_keys(client, "hmc-sleep-staging/1.1/recordings/")
    names = sorted(Path(k).stem for k in keys
                   if k.endswith(".edf") and "_sleepscoring" not in k)
    pre = "hmc-sleep-staging/1.1/recordings/"
    return [(f"hmc-{n}", pre + f"{n}.edf", pre + f"{n}_sleepscoring.edf")
            for n in names if pre + f"{n}_sleepscoring.edf" in keys]


def draw_u(rng):
    pct = rng.uniform(0, 100)
    return float(np.interp(pct, np.arange(0, 101, 5), MUSE_ONSET_PCT))


def process(dataset, subject, psg_key, ann_key, seed, tmp):
    import mne
    mne.set_log_level("ERROR")
    client = s3()
    rng = np.random.default_rng(seed)
    with tempfile.TemporaryDirectory(dir=tmp) as d:
        psg = os.path.join(d, "psg.edf")
        ann = os.path.join(d, "ann.edf")
        client.download_file(BUCKET, ann_key, ann)
        annots = mne.read_annotations(ann)
        desc = [str(s) for s in annots.description]
        n2 = annots.onset[np.array([s in ("Sleep stage 2", "Sleep stage N2")
                                    for s in desc], dtype=bool)]
        if not len(n2):
            return None, "no N2"
        first_n2 = float(n2.min())
        u = None
        lights = annots.onset[np.array(["lights off" in s.lower()
                                        for s in desc], dtype=bool)]
        if len(lights):
            cand = first_n2 - float(lights.min())
            if 60.0 <= cand <= MAX_U_S:
                u = cand
        if u is None:
            u = draw_u(rng)
        u = min(u, first_n2)
        if u < 2 * WIN_S:
            return None, "N2 too close to the start"
        client.download_file(BUCKET, psg_key, psg)
        raw = mne.io.read_raw_edf(psg, preload=False)
        start = first_n2 - u
        raw.crop(tmin=max(0.0, start - 2.0),
                 tmax=min(raw.times[-1], first_n2 + 2.0))
        raw.load_data()
        names = raw.ch_names
        if dataset == "sleepedf":
            need = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]
        else:
            need = ["EEG F4-M1", "EEG C3-M2", "EOG E1-M2", "EOG E2-M2"]
        if any(n not in names for n in need):
            return None, f"missing channels ({names})"
        raw.pick(need)
        if raw.info["sfreq"] > SFREQ:                      # anti-alias
            raw.filter(None, 0.45 * SFREQ, verbose="ERROR")
        raw.resample(SFREQ)
        x = raw.get_data() * 1e6                           # V -> uV
        t0 = raw.first_time if hasattr(raw, "first_time") else 0.0
        offset = int(round((start - max(0.0, start - 2.0)) * SFREQ))
        if dataset == "sleepedf":
            fpz_cz, pz_oz, eog = x
            muse = np.stack([-fpz_cz, eog / 2, -eog / 2, pz_oz])
        else:
            f4, c3, e1, e2 = x
            muse = np.stack([-f4, e1 - f4, e2 - f4, -c3])
        n_win = int(u // WIN_S)
        n_t = int(WIN_S * SFREQ)
        seg = muse[:, offset:offset + n_win * n_t]
        n_win = seg.shape[1] // n_t
        if n_win < 2:
            return None, "too short after cropping"
        seg = seg[:, :n_win * n_t]
        X = seg.reshape(4, n_win, n_t).transpose(1, 0, 2)
        ends = (np.arange(n_win) + 1) * WIN_S
        y = np.minimum(u - ends, CAP_S).clip(0, None)
        del t0
        return dict(X=np.clip(X, -6e4, 6e4).astype(np.float16),
                    y=y.astype(np.float32),
                    onset=(np.arange(n_win) * WIN_S).astype(np.float64),
                    subject=subject, dataset=dataset, u=u), "ok"


def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--datasets", default="sleepedf,hmc")
    ap.add_argument("--out", default="external_windows.npz")
    ap.add_argument("--limit", type=int, default=0,
                    help="recordings per dataset (0 = all); for testing")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--tmp", default=None, help="scratch folder for downloads")
    args = ap.parse_args()

    client = s3()
    jobs = []
    for ds in args.datasets.split(","):
        recs = recordings(client, ds)
        if args.limit:
            recs = recs[:args.limit]
        log(f"{ds}: {len(recs)} recordings")
        jobs += [(ds, *r) for r in recs]

    results, skipped = [], {}
    with cf.ThreadPoolExecutor(args.workers) as pool:
        futs = {pool.submit(process, ds, sub, pk, ak, i, args.tmp): (ds, pk)
                for i, (ds, sub, pk, ak) in enumerate(jobs)}
        for n, fut in enumerate(cf.as_completed(futs), 1):
            ds, pk = futs[fut]
            try:
                res, why = fut.result()
            except Exception as err:  # noqa: BLE001 - keep going
                res, why = None, f"error {err}"
            if res is None:
                skipped[Path(pk).name] = why
            else:
                results.append(res)
            if n % 10 == 0 or n == len(jobs):
                log(f"{n}/{len(jobs)} done, {len(results)} kept, "
                    f"{len(skipped)} skipped")

    if not results:
        raise SystemExit(f"nothing kept; skipped: {skipped}")
    results.sort(key=lambda r: (r["dataset"], r["subject"], r["u"]))
    X = np.concatenate([r["X"] for r in results])
    np.savez_compressed(
        args.out, X=X,
        y=np.concatenate([r["y"] for r in results]),
        rec=np.concatenate([np.full(len(r["y"]), i)
                            for i, r in enumerate(results)]),
        onset=np.concatenate([r["onset"] for r in results]),
        subject=np.concatenate([np.full(len(r["y"]), r["subject"])
                                for r in results]),
        dataset=np.concatenate([np.full(len(r["y"]), r["dataset"])
                                for r in results]))
    for ds in sorted({r["dataset"] for r in results}):
        rs = [r for r in results if r["dataset"] == ds]
        log(f"{ds}: {len(rs)} recordings, {sum(len(r['y']) for r in rs)} "
            f"windows, median stream {np.median([r['u'] for r in rs]):.0f} s")
    for name, why in sorted(skipped.items())[:20]:
        log(f"skipped {name}: {why}")
    log(f"wrote {args.out}: {X.shape[0]} windows, "
        f"{os.path.getsize(args.out) / 1e6:.0f} MB")


if __name__ == "__main__":
    main()

YASA teacher labels for the staging output: YASA's wake / drowsy / N2+
probabilities per 30 s epoch of every Muse recording, computed offline
(`scripts/yasa_check.py`, `scripts/make_yasa_teacher.py`). Embedded here, so
YASA is never installed on Colab.

In [ ]:
import base64
from pathlib import Path
TEACHER_B64 = """
...
"""
teacher = PROJECT / "data" / "yasa_teacher.npz"
teacher.parent.mkdir(parents=True, exist_ok=True)
teacher.write_bytes(base64.b64decode("".join(TEACHER_B64.split())))
print(f"wrote {teacher} ({teacher.stat().st_size / 1024:.0f} KB)")

## Install dependencies (keeps Colab's CUDA torch)

In [ ]:
!pip install -q -r competition/requirements.txt

neuralbench asks for its folders (CACHE_DIR, SAVE_DIR, DATA_DIR, W&B) on first
use. The harness overrides those paths with its own data folder, so write the
config here and skip the interactive prompt.

In [ ]:
import json
nb_config = Path.home() / ".neuralbench" / "config.json"
if not nb_config.exists():
    base = BASE / "neuralbench"
    nb_config.parent.mkdir(parents=True, exist_ok=True)
    nb_config.write_text(json.dumps({
        "USER": "colab", "ENTITY_NAME": "colab", "PROJECT_NAME": "neuralbench",
        "SLURM_PARTITION": "", "SLURM_CONSTRAINT": "", "N_CPUS": 2,
        "CLUSTER": None,
        "CACHE_DIR": str(base / "cache"), "SAVE_DIR": str(base / "save"),
        "DATA_DIR": str(base / "data"), "WANDB_HOST": "",
    }, indent=2))
    for sub in ("cache", "save", "data"):
        (base / sub).mkdir(parents=True, exist_ok=True)
print(nb_config.read_text())

## Muse data: from the Drive cache, else download from NEMAR

In [ ]:
DATA = PROJECT / "competition/tracks/sleep_onset/data"
CACHE = DRIVE_DIR / "muse_data.tar" if USE_DRIVE else None

if not (DATA / "neural_compet").exists() and CACHE is not None and CACHE.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    !tar -xf "{CACHE}" -C "{DATA}"
    print("data restored from Drive")

!benchopt prepare competition/tracks/sleep_onset -d Interaxon2026Muse

if CACHE is not None and not CACHE.exists():
    !tar -cf "{CACHE}" -C "{DATA}" neural_compet
    print("data cached in Drive:", CACHE)

## External PSG windows (one-off; only with `RUN_EXTERNAL_PREP = True`)

Streams Sleep-EDF (153 nights) and HMC (151 nights) from PhysioNet's open S3
mirror one recording at a time and cuts their last minutes before the first
N2 into pseudo-Muse windows (see `scripts/prep_external.py`). Then: **Save
Version**, open the version's Output, **New Dataset** from
`external_windows.npz`, and attach that dataset to this notebook ("Add
Input"); set `RUN_EXTERNAL_PREP = False` again.

In [ ]:
if RUN_EXTERNAL_PREP and not EXT_OUT.exists():
    EXT_OUT.parent.mkdir(parents=True, exist_ok=True)
    !pip install -q boto3
    !cd "{PROJECT}" && python scripts/prep_external.py --out "{EXT_OUT}" --tmp "{BASE}" --workers 8
elif EXT_FILE is not None:
    print("using", EXT_FILE)

## Smoke test on synthetic data (seconds; the score is meaningless)

In [ ]:
!SLEEP_N_SEEDS=1 python scripts/build_submission.py --solver submission/cnn_gru.py --label smoke_cnn --force 2>&1 | grep -E "CNN-GRU\\]|Error|Ready"
!python scripts/build_submission.py --solver submission/gbm.py --label smoke_gbm --force 2>&1 | grep -E "GBM\\] (ensemble|[0-9]+ train)|Error|Ready"

## Experiments

Every run is scored two ways:

* **val W-bMAE**: 15% of the training *sleepers*, held out and identical for
  every run and both solvers. These sleepers are never trained on, so this
  approximates the **unseen** half of the sealed test.
* **test W-bMAE**: the 40 warm-up recordings (sleepers **seen** in training),
  scored inference-only as on Codabench.

**Round 8** (default, `ROUND = 8`) is a sweep plus a merged ensemble, all
cross-validated on all 540 recordings (one seed each, ~11 min per run):

| Run | Change from the combo (autoencoder + hand-crafted + preprocessing v2) |
|---|---|
| `r8_combo` | none (seed 0; round 6's combo was seed 1) |
| `r8_gru192` | GRU 192 units instead of 128 |
| `r8_ae64` | autoencoder code 64 instead of 32 |
| `r8_ema30` / `r8_ema120` | short-term trend over 30 s / 120 s instead of 60 s |
| `r8_sigma3` | wider soft labels (3 bins instead of 1.5) |
| `r8_chunk240` | backpropagation through 20 min instead of 10 |
| `r8_lr5e4` | learning rate 5e-4, 40 epochs |
| `r8_ae_only` | autoencoder replaces the hand-crafted features |
| `r8_prep_only` | no autoencoder |
| `r8_ext05` / `r8_ext1` | + external PSG windows (Sleep-EDF, HMC), loss weight 0.5 / 1 (only when `EXT_FILE` is set) |

Every run saves its **out-of-fold distributions** (`*_oof.npz`). The merge
cell then picks the best weighted mix of all runs on them and writes
`r8_merged.zip`. A single run beats the combo only if its `oof540` is lower
by more than ~0.5 s (one seed's noise); the merged ensemble is judged by its
honest fold-5 check.

External data (one-off): set `RUN_EXTERNAL_PREP = True` for one session (see
the External PSG section), save its output as a Kaggle Dataset and attach it;
later sessions pick it up automatically.

**Round 7** (`ROUND = 7`) built the candidate final submission: the
round 6 winner (**combo**: autoencoder + hand-crafted features + preprocessing
v2) as a **2-seed cross-validated ensemble**, 2 models per fold = 10 models,
trained on all 540 recordings (`SLEEP_CV_SEEDS=2`, about twice a normal CV
run).

Round 6 (out-of-fold over 540 recordings; seed 0 from round 5):

| Build | seed 0 | seed 1 |
|---|---|---|
| base | 42.4 | 41.9 |
| autoencoder + hand | 41.0 | 41.6 |
| **combo** (+ preprocessing v2) | – | **40.8** |

A new seed moves a build by about 0.5 s; averaging seeds removes that noise.
The log prints each seed alone and the 10-model ensemble, all out-of-fold,
so the gain from the second seed is measured directly.

**Round 6** (`ROUND = 6`) checked round 5's winner and combines it
with the preprocessing fix. Round 5 (seed 0, out-of-fold over 540
recordings): base 42.4, preprocessing v2 41.5, autoencoder only 41.7,
**autoencoder + hand-crafted 41.0** (better in all 5 folds). Round 6 reruns
with **seed 1**, all cross-validated on 540 recordings:

| Run | Settings | Question |
|---|---|---|
| `r4_base_cv_s1` | base features | how much does a new seed move `oof540`? |
| `r5_ae_hand_cv_s1` | + autoencoder (`SLEEP_AE=2`) | does the 1.4 s gain hold on another seed? |
| `r6_combo_cv_s1` | + autoencoder + preprocessing v2 (`SLEEP_PREP=2`) | do the two gains add up? |

Seed 1 changes only the models' initialisation and batch order; the folds
stay the same, so the three runs are paired with each other (Kaggle runs
are deterministic: `r4_base_cv_s0` gave identical numbers twice).

**Round 5** (`ROUND = 5`) tested two things, all as cross-validated
builds on all 540 recordings, compared by `oof540`:

| Run | What changes |
|---|---|
| `r4_move_cv` (or `r4_base_cv`) | baseline: round 4's winner, set by `R5_FEATS` |
| `r5_prep_cv` | **preprocessing v2**: band powers skip 48-52 Hz and use a Welch estimate |
| `r5_ae_cv` | **autoencoder features** *replace* the hand-crafted ones |
| `r5_ae_hand_cv` | autoencoder features *added to* the hand-crafted ones |

* **Preprocessing v2** (`SLEEP_PREP=2`): the raw recordings carry 50 Hz mains
  noise (in the median recording 3.5x the neighbouring frequencies, up to
  5000x), inside the 30-60 Hz band that also sits in the denominator of every
  relative band power. v2 skips 48-52 Hz and averages 2 s Hann segments
  (Welch) instead of one 5 s periodogram, for steadier band powers.
* **Autoencoder** (`SLEEP_AE`): a denoising MLP autoencoder is trained, without
  labels, on each fold's training windows: input = log spectrum in 27
  log-spaced bins (0.4-47 Hz) on the 4 channels and AF7-AF8, code = 32 dims.
  Its frozen encoder then feeds the GRU in place of (`=1`) or next to (`=2`)
  the hand-crafted features. The log's `[CNN-GRU AE ...]` lines give its
  held-out reconstruction error (1.0 = no better than the mean).

The baseline has exactly round 4's settings and label, so on Colab with Drive
it is restored from round 4 instead of retrained; on Kaggle (no saved
results between sessions) it is rebuilt in the same session, which also
pairs the comparison. `R5_FEATS = 2`: in round 4 the movement features
(`r4_move_cv`, 42.1 s out-of-fold) tied with the base (`r4_base_cv`, 42.4 s;
same on the 60 validation recordings, false "near" calls unchanged), so
round 5 builds on the simpler feature set. About 4 CV runs: fits one Kaggle
"Save & Run All" session.

**Round 4** (`ROUND = 4`) tested movement / arousal features. The
model's main remaining error is calling "sleep is near" minutes too early,
when a drowsy sleeper later wakes back up, and movements predict those false
calls. `SLEEP_FEATS=3` adds per-window movement markers (clipped samples, peak
amplitude, line length) and a fading memory of recent movements (~1 and
~5 min).

Both runs are **cross-validated builds on all 540 recordings** (like
`gru_cv5_all`), so they are compared on the out-of-fold W-bMAE over all 540
recordings (`oof540` in the summary), far more reliable than the 60
validation recordings. Run in the same session, so they are paired:

| Run | Features |
|---|---|
| `r4_base_cv` | drowsiness features (= `gru_cv5_all`) |
| `r4_move_cv` | + movement features |

Each run is a complete submission: the winner's ZIP (Drive `results/`, or
`dist/`) can be uploaded to the sealed phase as is. `test` is blank for these
runs, since they train on the warm-up test recordings.

**Round 3** (`ROUND = 3`) tested the staging output distilled from
YASA, on top of round 2's best setup (`r2_feats_extra`: no CNN, drowsiness
features, extra recordings), 3 seeds each:

| Run | Staging loss weight |
|---|---|
| `r3_base` | 0 (off; = `r2_feats_extra`) |
| `r3_stage05` | 0.5 |
| `r3_stage1` | 1.0 |

The staging output predicts YASA's wake / drowsy (N1 + REM) / N2+
probabilities for each window. It is used only in training, to shape what
the GRU tracks; predictions still come from the time-to-N2 output alone, and
the submission format is unchanged. The log's `stage CE` is its loss.

**Round 2** (`ROUND = 2`) tested the three changes made after round 1, all on
the GRU without the CNN, 3 seeds each:

| Run | Extra recordings | Drowsiness and trend features |
|---|---|---|
| `r2_base` | no | no (= round 1's `cnn_gru_no_cnn`) |
| `r2_extra` | yes | no |
| `r2_feats` | no | yes |
| `r2_feats_extra` | yes | yes (the new default) |

* **Extra recordings**: NeuralBench's own validation sleepers from the public
  training release (about 88 recordings), which the harness's train loader
  leaves out.
* **Drowsiness features**: slow eye-movement power on AF7−AF8, spectral entropy
  and edge, plus each feature's change over the last ~1 min and its trend
  against the recording's mean.

`ROUND = 1` reruns the bottleneck experiments (CNN vs no CNN, learning curve,
GBM). Finished runs are saved in Drive and skipped on a re-run, so after a
disconnect just **Run all** again.

`RUN_PRODUCTION = True` also builds the submission for the sealed phase with
the settings in `PRODUCTION_ENV`: a **cross-validated ensemble trained on all
540 public recordings** (the harness's training set, the 88 extra recordings
and the 40 warm-up test recordings). Each of its 5 models holds out a
different fifth of the sleepers for early stopping, so together they train on
everything. Its `val` in the summary is out-of-fold on the usual 60
validation recordings (each predicted by the model that never saw that
sleeper), so it stays roughly comparable with the experiments. Its `test` is
meaningless, since the warm-up test recordings are in its training data.
To build only the submission, also set `RUN_EXPERIMENTS = False`. Finished
experiments are restored from Drive anyway, but without Drive they would
retrain.

In [ ]:
import os, shutil

ROUND = 8
R5_FEATS = 2                # round 5 base features: 2 (round 4: movement features, 3, were a tie)
SEEDS = [0, 1, 2]
RUN_EXPERIMENTS = True      # the experiment runs of ROUND (False: skip them)
RUN_PRODUCTION = False      # build the sealed-phase submission
PRODUCTION_LABEL = "combo_cv5x2_all"
# 5-fold cross-validated ensemble on all 540 public recordings; solver
# defaults: no CNN, feats 2, extra data, no staging output. If round 3's
# staging output wins, add e.g. "SLEEP_STAGE_WEIGHT": "0.5" (and change the
# label, so Drive does not restore the earlier build).
PRODUCTION_ENV = {"SLEEP_CV_FOLDS": "5", "SLEEP_CV_SEEDS": "2", "SLEEP_USE_TEST": "1",
                  "SLEEP_FEATS": "2", "SLEEP_AE": "2", "SLEEP_PREP": "2"}

GBM, CNN = "submission/gbm.py", "submission/cnn_gru.py"
# Round 1 settings spelled out, since the solver defaults changed after it.
V1 = {"SLEEP_N_SEEDS": "1", "SLEEP_ADAPT": "1", "SLEEP_USE_CNN": "1",
      "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "0"}
GRU = {"SLEEP_N_SEEDS": "1", "SLEEP_ADAPT": "1", "SLEEP_USE_CNN": "0"}
# Round 8: sweep around the combo (autoencoder + hand-crafted features +
# preprocessing v2), 5-fold CV on all 540 recordings, one seed each.
C = {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "2",
     "SLEEP_AE": "2", "SLEEP_PREP": "2"}
R8 = [
    ("r8_combo",         CNN, C),
    ("r8_gru192",        CNN, {**C, "SLEEP_GRU_HIDDEN": "192"}),
    ("r8_ae64",          CNN, {**C, "SLEEP_AE_DIM": "64"}),
    ("r8_ema30",         CNN, {**C, "SLEEP_EMA_S": "30"}),
    ("r8_ema120",        CNN, {**C, "SLEEP_EMA_S": "120"}),
    ("r8_sigma3",        CNN, {**C, "SLEEP_LABEL_SIGMA": "3"}),
    ("r8_chunk240",      CNN, {**C, "SLEEP_CHUNK": "240"}),
    ("r8_lr5e4",         CNN, {**C, "SLEEP_LR": "5e-4", "SLEEP_EPOCHS": "40"}),
    ("r8_ae_only",       CNN, {**C, "SLEEP_AE": "1"}),
    ("r8_prep_only",     CNN, {**C, "SLEEP_AE": "0"}),
]
if EXT_FILE is not None:
    R8 += [("r8_ext05", CNN, {**C, "SLEEP_EXTERNAL": str(EXT_FILE), "SLEEP_EXT_WEIGHT": "0.5"}),
           ("r8_ext1",  CNN, {**C, "SLEEP_EXTERNAL": str(EXT_FILE), "SLEEP_EXT_WEIGHT": "1.0"})]
EXPERIMENTS = {
    1: [  # (name,           solver, env)
        ("gbm_time",         GBM, {"SLEEP_GBM_FEATURES": "time"}),
        ("gbm_all",          GBM, {}),
        ("gbm_frac25",       GBM, {"SLEEP_TRAIN_FRAC": "0.25"}),
        ("gbm_frac50",       GBM, {"SLEEP_TRAIN_FRAC": "0.5"}),
        ("cnn_gru_full",     CNN, V1),
        ("cnn_gru_no_cnn",   CNN, {**V1, "SLEEP_USE_CNN": "0"}),
        ("cnn_gru_frac25",   CNN, {**V1, "SLEEP_TRAIN_FRAC": "0.25"}),
        ("cnn_gru_frac50",   CNN, {**V1, "SLEEP_TRAIN_FRAC": "0.5"}),
    ],
    2: [
        ("r2_base",          CNN, {**GRU, "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "0"}),
        ("r2_extra",         CNN, {**GRU, "SLEEP_FEATS": "1", "SLEEP_EXTRA_DATA": "1"}),
        ("r2_feats",         CNN, {**GRU, "SLEEP_FEATS": "2", "SLEEP_EXTRA_DATA": "0"}),
        ("r2_feats_extra",   CNN, {**GRU, "SLEEP_FEATS": "2", "SLEEP_EXTRA_DATA": "1"}),
    ],
    4: [  # cross-validated all-data builds, one run each (see markdown)
        ("r4_base_cv",       CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "2"}),
        ("r4_move_cv",       CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "3"}),
    ],
    5: [  # cross-validated all-data builds, one run each (see markdown)
        # Baseline = round 4's run with the same settings (restored if saved).
        ("r4_move_cv" if R5_FEATS == 3 else "r4_base_cv", CNN,
                             {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": str(R5_FEATS)}),
        ("r5_prep_cv",       CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": str(R5_FEATS), "SLEEP_PREP": "2"}),
        ("r5_ae_cv",         CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_AE": "1"}),
        ("r5_ae_hand_cv",    CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": str(R5_FEATS), "SLEEP_AE": "2"}),
    ],
    6: [  # cross-validated all-data builds, seed 1 (see markdown)
        ("r4_base_cv",       CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "2"}),
        ("r5_ae_hand_cv",    CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "2", "SLEEP_AE": "2"}),
        ("r6_combo_cv",      CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_USE_TEST": "1", "SLEEP_FEATS": "2", "SLEEP_AE": "2", "SLEEP_PREP": "2"}),
    ],
    7: [  # the candidate final build: combo, 5 folds x 2 seeds (see markdown)
        ("r7_combo_cv2",     CNN, {"SLEEP_CV_FOLDS": "5", "SLEEP_CV_SEEDS": "2", "SLEEP_USE_TEST": "1",
                                   "SLEEP_FEATS": "2", "SLEEP_AE": "2", "SLEEP_PREP": "2"}),
    ],
    8: R8,
    3: [
        ("r3_base",          CNN, {**GRU, "SLEEP_STAGE_WEIGHT": "0"}),
        ("r3_stage05",       CNN, {**GRU, "SLEEP_STAGE_WEIGHT": "0.5"}),
        ("r3_stage1",        CNN, {**GRU, "SLEEP_STAGE_WEIGHT": "1.0"}),
    ],
}[ROUND]
if ROUND in (4, 5, 7, 8):
    SEEDS = [0]
if ROUND == 6:
    SEEDS = [1]     # each run is already a 5-model ensemble over 540 recordings
RUNS = [(f"{name}_s{seed}", solver, {**env, "SLEEP_SEED": str(seed)})
        for name, solver, env in EXPERIMENTS for seed in SEEDS
        if RUN_EXPERIMENTS]
if RUN_PRODUCTION:
    RUNS.append((PRODUCTION_LABEL, CNN, PRODUCTION_ENV))

BUILD = PROJECT / "build"
RESULTS = (DRIVE_DIR / "results" if USE_DRIVE
           else KAGGLE_RESULTS if ON_KAGGLE else None)

def run_files(label):
    return [p for p in BUILD.glob(f"{label}_*") if p.is_file()] + \
           [BUILD / f"{label}.log"]

def restore(label):
    """Copy a finished run back from Drive; True if it was there."""
    if RESULTS is None or not (RESULTS / f"{label}_eval.parquet").exists():
        return False
    for name in ("eval.parquet", "train.parquet", "valpred.npz", "oof.npz"):
        src = RESULTS / f"{label}_{name}"
        if src.exists():
            shutil.copy(src, BUILD)
    if (RESULTS / label).is_dir():          # weights, for merging
        shutil.copytree(RESULTS / label, BUILD / label, dirs_exist_ok=True)
    if (RESULTS / f"{label}.log").exists():
        shutil.copy(RESULTS / f"{label}.log", BUILD)
    return True

def build(label, solver, env_extra):
    env = {**os.environ, **env_extra}
    log_path = BUILD / f"{label}.log"
    cmd = ["python", "scripts/build_submission.py", "--data", "muse",
           "--solver", solver, "--label", label, "--force"]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if any(k in line for k in ("[CNN-GRU", "[GBM]", "Error", "Ready")):
                print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{label} failed; see {log_path}")
    if RESULTS is not None:
        RESULTS.mkdir(parents=True, exist_ok=True)
        shutil.copytree(BUILD / label, RESULTS / label, dirs_exist_ok=True)
        for f in run_files(label):
            if f.exists():
                shutil.copy(f, RESULTS)
        shutil.copy(PROJECT / "dist" / f"{label}.zip", RESULTS)

if RUNS and not torch.cuda.is_available() and not all(
        (RESULTS is not None and (RESULTS / f"{l}_eval.parquet").exists()) for l, _, _ in RUNS):
    raise RuntimeError("Training needs a GPU runtime (Kaggle: GPU T4 x2). For the "
                       "external-data prep alone set RUN_EXPERIMENTS = False.")
for label, solver, env_extra in RUNS:
    if restore(label):
        print(f"===== {label}: restored from Drive")
        continue
    print(f"===== {label}: {env_extra}")
    build(label, solver, env_extra)

### Merged ensemble (round 8)

`scripts/merge_builds.py` mixes the finished CV builds (plus `r7_combo_cv2_s0`
and `r6_combo_cv_s1` if their results are present) by greedy selection on
their out-of-fold distributions, tunes a quantile per elapsed-time bucket,
and writes one submission, `r8_merged.zip`. Its first line of interest is
the **honest check** (selection fitted on folds 1-4, scored on fold 5);
then the final out-of-fold score of the ensemble vs the best single build.

In [ ]:
MERGED = "r8_merged"
_have = ROUND == 8 and any((BUILD / f"{l}_oof.npz").exists() or restore(l) for l, _, _ in RUNS)
if ROUND == 8 and not _have:
    print("no finished round 8 runs to merge")
if _have:
    for extra in ("r7_combo_cv2_s0", "r6_combo_cv_s1"):
        restore(extra)
    cands = [l for l, _, _ in RUNS] + ["r7_combo_cv2_s0", "r6_combo_cv_s1"]
    cands = [l for l in cands if (BUILD / f"{l}_oof.npz").exists()]
    print("merging:", cands)
    log_path = BUILD / f"{MERGED}.log"
    with open(log_path, "w") as log:
        proc = subprocess.Popen(["python", "scripts/merge_builds.py", "--label", MERGED, *cands],
                                cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if not line.startswith(("+", "[")) or "Error" in line:
                print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"merge failed; see {log_path}")
    if RESULTS is not None:
        RESULTS.mkdir(parents=True, exist_ok=True)
        shutil.copytree(BUILD / MERGED, RESULTS / MERGED, dirs_exist_ok=True)
        for f in run_files(MERGED):
            if f.exists():
                shutil.copy(f, RESULTS)
        shutil.copy(PROJECT / "dist" / f"{MERGED}.zip", RESULTS)

## 1. Summary

Mean ± std over seeds. `gap` = test − val: how much easier seen sleepers are
than unseen ones. The constant row is a 35 s constant (the published floor),
scored on the same validation recordings.

In [ ]:
import re
import numpy as np
import pandas as pd

EDGES = np.array([0, 40, 90, 300, 600.0])
WEIGHTS = np.array([10, 5, 3, 1.0])

def range_of(y):
    return np.clip(np.digitize(y, EDGES[1:-1]), 0, len(WEIGHTS) - 1)

def wbmae(y, p):
    """W-bMAE of one recording, as benchmark_utils.metrics.binned_mae."""
    keep = (y >= EDGES[0]) & (y <= EDGES[-1])      # out-of-range targets ignored
    y, p = y[keep], p[keep]
    r, err = range_of(y), np.abs(p - y)
    present = np.unique(r)
    return sum(WEIGHTS[b] * err[r == b].mean() for b in present) / WEIGHTS[present].sum()

def score(df, col="pred"):
    return np.mean([wbmae(g.y.values, g[col].values) for _, g in df.groupby("record")])

def valpred(label):
    f = BUILD / f"{label}_valpred.npz"
    if not f.exists():
        return None
    d = np.load(f)
    return pd.DataFrame({k: d[k] for k in ("record", "subject", "position", "y", "pred")})

def experiment(label):
    return re.sub(r"_s\d+$", "", label)

rows = []
for label, _, _ in RUNS:
    eval_file = BUILD / f"{label}_eval.parquet"
    if not eval_file.exists():
        continue
    d = pd.read_parquet(eval_file).filter(like="objective_").iloc[-1]
    log = (BUILD / f"{label}.log").read_text() if (BUILD / f"{label}.log").exists() else ""
    val = re.findall(r"ensemble val W-bMAE ([\d.]+)", log)
    trained_on_test = "test W-bMAE is meaningless" in log
    oof = re.findall(r"out-of-fold W-bMAE ([\d.]+) s on all", log)
    rows.append({"experiment": experiment(label), "label": label,
                 # Cross-validated builds: out-of-fold over all recordings.
                 "oof540": float(oof[-1]) if oof else np.nan,
                 "val": float(val[-1]) if val else np.nan,
                 # A build trained on the warm-up test recordings has no test score.
                 "test": np.nan if trained_on_test else d["objective_wbmae_recording_mean"],
                 "test bMAE": d["objective_bmae"], "test MAE": d["objective_mae"]})
runs = pd.DataFrame(rows, columns=["experiment", "label", "oof540", "val",
                                   "test", "test bMAE", "test MAE"])

summary = runs.groupby("experiment", sort=False).agg(
    seeds=("label", "size"),
    oof540=("oof540", "mean"),
    val=("val", "mean"), val_std=("val", "std"),
    test=("test", "mean"), test_std=("test", "std"))
summary["gap (test-val)"] = summary.test - summary.val

any_val = next((valpred(l) for l in runs.label if valpred(l) is not None), None)
if any_val is not None:
    const = any_val.assign(pred=35.0)
    summary.loc["constant_35s"] = {"seeds": 0, "val": score(const)}
summary.round(1)

## 2. Learning curve (round 1 only): val W-bMAE vs share of training sleepers

In [ ]:
import matplotlib.pyplot as plt

FAMILIES = {   # name: (color, {train fraction: experiment})
    "CNN-GRU": ("#2a78d6", {0.25: "cnn_gru_frac25", 0.5: "cnn_gru_frac50", 1.0: "cnn_gru_full"}),
    "GBM":     ("#eb6834", {0.25: "gbm_frac25", 0.5: "gbm_frac50", 1.0: "gbm_all"}),
}
if not any(e in summary.index for _, pts in FAMILIES.values() for e in pts.values()):
    print("No learning-curve runs in this round (ROUND = 1 has them).")
else:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for name, (color, points) in FAMILIES.items():
        xs = [f for f, e in points.items() if e in summary.index]
        if not xs:
            continue
        m = np.array([summary.loc[points[f], "val"] for f in xs])
        s = np.nan_to_num(np.array([summary.loc[points[f], "val_std"] for f in xs]))
        ax.plot(xs, m, "-o", color=color, lw=2, ms=8, label=name)
        ax.fill_between(xs, m - s, m + s, color=color, alpha=0.15, lw=0)
        ax.annotate(name, (xs[-1], m[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color="#333")
    if "constant_35s" in summary.index:
        ax.axhline(summary.loc["constant_35s", "val"], color="#888", lw=1, ls="--")
        ax.annotate("constant 35 s", (0.25, summary.loc["constant_35s", "val"]),
                    xytext=(0, 4), textcoords="offset points", color="#666", fontsize=9)
    ax.set_xscale("log", base=2)
    ax.set_xticks([0.25, 0.5, 1.0], ["25%", "50%", "100%"])
    ax.set_xlim(0.22, 1.35)
    ax.set_xlabel("share of training sleepers used")
    ax.set_ylabel("val W-bMAE, unseen sleepers (s, lower is better)")
    ax.grid(axis="y", color="#e5e5e5", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, loc="best")
    plt.tight_layout(); plt.show()

## 3. Where does the error come from?

Per range of the true time-to-N2, on the validation recordings (all seeds
pooled): the MAE, and how many seconds of the W-bMAE each range contributes
(the contributions sum to the score). The range with the largest
contribution is where improvements pay off most. The 0–40 s range carries
weight 10, so it usually dominates.

In [ ]:
LABELS = ["0-40 s", "40-90 s", "90-300 s", "300-600 s"]

def breakdown(df):
    mae = {b: [] for b in range(4)}
    contrib = np.zeros(4)
    n_rec = 0
    for _, g in df.groupby(["seed", "record"]):
        g = g[(g.y >= EDGES[0]) & (g.y <= EDGES[-1])]
        r = range_of(g.y.values)
        err = np.abs(g.pred.values - g.y.values)
        present = np.unique(r)
        for b in present:
            mae[b].append(err[r == b].mean())
            contrib[b] += WEIGHTS[b] * err[r == b].mean() / WEIGHTS[present].sum()
        n_rec += 1
    out = {f"MAE {LABELS[b]}": np.mean(mae[b]) if mae[b] else np.nan for b in range(4)}
    out.update({f"share {LABELS[b]}": contrib[b] / n_rec for b in range(4)})
    return out

rows = {}
for exp in summary.index:
    parts = [valpred(l).assign(seed=l) for l in runs.label[runs.experiment == exp]
             if valpred(l) is not None]
    if parts:
        rows[exp] = breakdown(pd.concat(parts))
if any_val is not None:
    rows["constant_35s"] = breakdown(any_val.assign(pred=35.0, seed=0))
pd.DataFrame(rows).T.round(1)

### Timing near onset

Median prediction (s) by true time to N2, on the validation recordings. A
well-timed model follows the diagonal: about 10 at 0–20 s, 50 at 40–60 s and
so on. In round 1 the model squeezed everything below 150 s into 20–60 s, so it
called onset too early.

In [ ]:
BINS = [-1, 20, 40, 60, 90, 150, 300, 601]
cols = {}
for exp in summary.index:
    parts = [valpred(l) for l in runs.label[runs.experiment == exp]
             if valpred(l) is not None]
    if parts:
        d = pd.concat(parts)
        cols[exp] = d.groupby(pd.cut(d.y, BINS), observed=True).pred.median()
timing = pd.DataFrame(cols)
timing.index = [f"{max(0, iv.left):.0f}-{min(600, iv.right):.0f} s" for iv in timing.index]
timing.index.name = "true time to N2"
timing.round(0)

## 4. Blend: GBM + CNN-GRU (round 1 only)

`w·GBM + (1−w)·CNN-GRU`, scored on the validation recordings. These
recordings also pick `w`, so the gain is slightly optimistic; a gain of only a
second or two is noise.

In [ ]:
def blend_curve(a_exp, b_exp):
    curves = []
    for seed in SEEDS:
        a, b = valpred(f"{a_exp}_s{seed}"), valpred(f"{b_exp}_s{seed}")
        if a is None or b is None:
            continue
        m = a.merge(b[["record", "position", "pred"]], on=["record", "position"],
                    suffixes=("_a", "_b"))
        ws = np.linspace(0, 1, 11)
        curves.append([score(m.assign(pred=w * m.pred_a + (1 - w) * m.pred_b))
                       for w in ws])
    if not curves:
        return None
    return pd.Series(np.mean(curves, axis=0), index=np.linspace(0, 1, 11).round(1))

blends = {f"gbm_all + {cnn}": blend_curve("gbm_all", cnn)
          for cnn in ("cnn_gru_full", "cnn_gru_no_cnn")}
blends = pd.DataFrame({k: v for k, v in blends.items() if v is not None})
if not blends.empty:
    blends.index.name = "w (GBM share)"
    for col in blends:
        best = blends[col].idxmin()
        print(f"{col}: best w = {best}, val W-bMAE {blends[col].min():.1f} "
              f"(GBM alone {blends[col][1.0]:.1f}, CNN-GRU alone {blends[col][0.0]:.1f})")
if blends.empty:
    print("No GBM runs in this round (ROUND = 1 has them).")
blends.round(1)

## 5. Decide

**Round 8.** Read the merge cell's output:

* **Honest check: ensemble clearly below the best single build on fold 5**
  (≳ 0.5 s): upload `r8_merged.zip` (results folder). It is a complete
  submission.
* **Not clearly better:** upload the best single build's ZIP by `oof540`.
* Sweep runs that beat `r8_combo` by more than ~0.5 s point to settings to
  keep in later builds; `r8_ext*` vs `r8_combo` is the external-data verdict.

**Round 7.** In the log of `r7_combo_cv2_s0`, compare the line `out-of-fold
W-bMAE ... on all 540 recordings` (the 10-model ensemble) with the two
`seed N alone` lines above it.

* **Ensemble below 40.8 (round 6's combo):** upload `r7_combo_cv2_s0.zip`.
  It is the complete final submission (10 models, all 540 recordings).
* **Not lower:** keep `r6_combo_cv_s1.zip`.

The streaming-check line also gives the inference time per window for
all 10 models.

**Round 6.** First the seed noise: `r4_base_cv_s1` vs round 5's 42.4.
Then `r5_ae_hand_cv_s1` vs `r4_base_cv_s1`:

* **Autoencoder gain holds (again ≳ 1 s lower):** it is real. Compare
  `r6_combo_cv_s1` with `r5_ae_hand_cv_s1`; take the combo if it is lower by
  more than the seed noise, otherwise the plain autoencoder build (fewer
  changes). Its ZIP is a complete submission. Ensembling the seed-0 and
  seed-1 builds of the winner (10 models) is the natural final step.
* **Gain gone:** round 5's 1.4 s was seed luck; stay with `gru_cv5_all`.

**Round 5.** Compare each run's `oof540` with the baseline (`r4_move_cv` or
`r4_base_cv`); about 0.5 s is a real difference on 540 recordings.

* **`r5_prep_cv` lower:** the mains-noise fix helps; keep `SLEEP_PREP=2`.
* **`r5_ae_cv` lower:** learned spectral features beat the hand-crafted ones.
  Unlikely with this little data, but it would point to bigger learned
  front-ends. If it is clearly worse, the hand-crafted features carry
  information the spectrum code misses (e.g. the trend and movement ones).
* **`r5_ae_hand_cv` lower:** the code adds something on top; keep `SLEEP_AE=2`.
* **Several win:** combine them in `PRODUCTION_ENV` (e.g. `"SLEEP_PREP": "2",
  "SLEEP_AE": "2"`, plus the round-4 `SLEEP_FEATS`) with a new
  `PRODUCTION_LABEL`, and build with `RUN_PRODUCTION = True`. Otherwise the
  winner's own ZIP (e.g. `r5_prep_cv_s0.zip`) is already a full submission.

**Round 4.** Compare `oof540` of `r4_move_cv` with `r4_base_cv`. With 540
recordings, about 0.5 s is already a real difference. In the error table, the
gain should show in the **90–300 s share** (false "near" calls).

* **`r4_move_cv` lower:** upload its ZIP (`r4_move_cv_s0.zip`) to the sealed
  phase. It is the complete submission.
* **Not lower:** keep `gru_cv5_all` (or `r4_base_cv_s0.zip`, the same setup).

**Round 3.** Compare `r3_stage05` and `r3_stage1` with `r3_base`, the same
way: validation first, then test, and only differences larger than about the
seeds' std count. The staging output targets the last 90 s, so look first at
the **0–40 s and 40–90 s shares** in the error table and at the **timing
table**: predictions for 40–60 s and 60–90 s should move up from the 30–40 s
they sat at in round 2.

* **A staging run wins:** add its weight to `PRODUCTION_ENV`, e.g.
  `"SLEEP_STAGE_WEIGHT": "0.5"`, and build the submission.
* **Neither beats `r3_base`:** YASA's labels add nothing the model didn't
  already have. Build without them; ANPHY's expert labels are the next try.

**Round 2.** Compare each run with `r2_base`, using validation first (the
unseen sleepers, 60 recordings) and test second. A change counts only if it
beats `r2_base` by more than about the seeds' std.

* **`r2_extra` better:** more data helps, as the round-1 learning curve
  suggested. Keep it on; external data (ANPHY, NSRR) is then worth a try.
* **`r2_feats` better, especially in the 0–40 and 40–90 s shares and in the
  timing table:** the drowsiness features help find onset. Keep them.
* **`r2_feats_extra` best:** build the submission with `RUN_PRODUCTION = True`
  (these are the defaults).
* **One change hurts:** turn it off in `PRODUCTION_ENV`, e.g.
  `"SLEEP_FEATS": "1"` or `"SLEEP_EXTRA_DATA": "0"`.

**Round 1** (`ROUND = 1`): CNN vs no CNN, the learning curve, GBM vs GRU; see
the reading guide at the top of the experiments.

Download all results (tables, logs, validation predictions) as one ZIP:

In [ ]:
import zipfile

out = PROJECT / "experiment_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for label in [l for l, _, _ in RUNS] + (["r8_merged"] if ROUND == 8 else []):
        for f in run_files(label):
            if f.exists():
                zf.write(f, f.name)
production_zip = PROJECT / "dist" / f"{PRODUCTION_LABEL}.zip"
if ON_COLAB:
    from google.colab import files
    files.download(str(out))
    if RUN_PRODUCTION and production_zip.exists():
        files.download(str(production_zip))
else:
    # Kaggle: everything in /kaggle/working is the notebook's Output.
    dest = Path("/kaggle/working") if ON_KAGGLE else PROJECT
    shutil.copy(out, dest)
    print("results ZIP:", dest / out.name)
    print("submission ZIPs:", sorted(str(p) for p in (RESULTS or PROJECT / "dist").glob("*.zip")))